# 손글씨 채점 AI

시험지 사진을 올리면 **반듯하게 펴서 칸을 자르고 → AI가 정답을 모른 채 읽고 → 신경망이 P(정답)을 계산해 판정하고 → 애매한 답만 교사에게 넘기고 → 빨간펜으로 채점된 시험지**까지 만듭니다.

| P(정답) | 판정 | 빨간펜 |
|---|---|---|
| T_HIGH(0.8) 이상 | **정답** | 동그라미 + 점수 |
| T_LOW(0.2) 이하 | **오답** | 빗금 |
| 그 사이, 또는 아래 규칙에 걸림 | **교사 확인** | "교사 채점 요함" 메모 |

확률과 상관없이 교사 확인: 칸 밖 글씨, 지운 자국·줄 그은 답, 순서만 바뀐 답(이신순), 혼동쌍만 다른 답(훈민정읍, 팔만대장견), 읽기 결과가 엇갈린 답, 한 글자 차이·정답 뒤에 말이 붙은 답(평양입니다). 교사가 등록한 오답(한양)은 그대로 오답.

### 같이 제출하는 것
- **시연 웹페이지(API 서버)**: 같은 채점 과정을 브라우저에서 해 볼 수 있습니다. 예시 답안지를 고르거나 휴대폰으로 찍어 올리면 빨간펜 채점 시험지와 성적표가 나옵니다. API 키는 서버에만 있어 다른 사람에게 넘어가지 않습니다.
- 이 노트북은 같은 과정을 단계별로 실행하며 **신경망 학습·검증과 그래프 분석**까지 보여 줍니다.

### 준비할 파일 (어떤 형식이든 됩니다)

| 무엇 | 형식 | 어디서 올림 |
|---|---|---|
| 정답표 (문항, 배점, 정답, 허용답안, 등록오답) | CSV, 엑셀, TSV | 1-3 |
| 명렬표 (반, 번호, 이름) | CSV, 엑셀 | 1-3 (엑셀 한 파일에 시트로 같이 넣어도 됨) |
| 답안지 | 사진(jpg, png, **아이폰 heic**), **여러 쪽 PDF 스캔**, **ZIP** | 4-1 |
| (사진 대신) 학생 답안 표 | CSV, 엑셀 | 1-3 |

### 단계
- **1단계 준비**: 라이브러리, 설정, 파일 올리기, 정답표, 특징 만들기
- **2단계 학습 데이터**: 학생 답과 읽기 실수를 흉내 낸 3,000개
- **3단계 신경망**: 학습(조기 종료) → 학습 곡선 → 판정 확인 → 기준값 찾기·득실 → 보정(ECE) → 시드 범위
- **4단계 채점**: 답안지 올리기 → 정렬·칸 자르기 → AI 읽기 → 학생 식별 → 채점·진단 → 그래프 → **빨간펜 채점 시험지**

> 숫자를 발표할 때: 3단계는 **시뮬레이션 기준**(3-6 시드 범위로), 4단계는 **올린 답안지 기준**이라고 구분하세요.

---
## 1단계. 준비
### 1-1. 라이브러리

In [ ]:
!pip -q install google-genai pillow-heif pymupdf fastapi uvicorn python-multipart anthropic   # -U를 빼야 Colab 기본 부품과 충돌하지 않음
!apt-get -qq install -y fonts-nanum > /dev/null

In [ ]:
import os, io, re, json, time, random, zipfile, shutil, unicodedata, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont, ImageOps
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
try:
    from IPython.display import display
except ImportError:
    display = print

# 그래프 한글 글꼴 + seaborn 테마 (seaborn 테마가 글꼴을 덮어쓰므로 테마에 같이 지정)
if not os.path.exists('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
FONT = 'sans-serif'
for _p in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    if os.path.exists(_p):
        font_manager.fontManager.addfont(_p)
        FONT = font_manager.FontProperties(fname=_p).get_name()
        GOTHIC_PATH = _p
        break
sns.set_theme(style='whitegrid', palette='deep', font=FONT, rc={'axes.unicode_minus': False, 'figure.dpi': 100})

# 빨간펜 채점 글씨용 손글씨 글꼴 (나눔펜, SIL 오픈 폰트 라이선스). 못 받으면 고딕으로 대신 씀
PEN_FONT_PATH = 'NanumPenScript-Regular.ttf'
if not os.path.exists(PEN_FONT_PATH):
    try:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumpenscript/'
                                   'NanumPenScript-Regular.ttf', PEN_FONT_PATH)
    except Exception:
        PEN_FONT_PATH = GOTHIC_PATH

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
for d in ('results', 'results/graded', 'inputs/data', 'inputs/sheets', 'work/pages', 'work/flat', 'crops'):
    os.makedirs(d, exist_ok=True)
JUDGE_ORDER = ['정답', '교사 확인', '오답']
JUDGE_COLOR = {'정답': '#2f9e44', '교사 확인': '#f59f00', '오답': '#e03131'}
print('준비 완료 · TensorFlow', tf.__version__)

### 1-2. 설정
판정 기준, 읽기 방식, 학습 방식을 한곳에서 정합니다. 처음엔 그대로 두고 실행해 보세요.
- **적응형 읽기**: 칸마다 2번 읽어 같으면 끝, 다르면 3번째까지 (5번씩 읽던 것보다 호출이 절반 가까이 줄어듦)
- **비용**: `PRICE_PER_1M_TOKENS`에 쓰는 모델의 100만 토큰당 가격(원)을 넣으면 4-7에서 답안지 1장당 비용을 계산합니다

In [ ]:
# ── 판정 기준 (교사가 정함) ──────────────────────────────────────
T_HIGH = 0.8                   # P(정답) ≥ T_HIGH → 정답   (3-4에서 데이터로 다시 찾음)
T_LOW = 0.2                    # P(정답) ≤ T_LOW → 오답,  그 사이 → 교사 확인
MIXED_READS_RULE = 'strict'    # 읽기가 엇갈릴 때: 'strict' 하나라도 엇갈리면 교사 확인 / 'lenient' 80% 미만이면 / 'off' 확률에 맡김
NEAR_MISS_TO_REVIEW = True     # 한 글자 차이, 정답 뒤에 말이 붙은 답 → 교사 확인
CASE_SENSITIVE = False         # 영어 대소문자 구분 (False면 DNA = dna)

# ── 손글씨 읽기 (Gemini) ─────────────────────────────────────────
USE_GEMINI = True              # False거나 키가 없으면 직접 입력 모드
GEMINI_MODEL = 'gemini-3.8-flash'   # AI Studio 모델 이름. 무료 등급이 없다는 에러가 나면 Dashboard의 Rate limits에서 무료 모델로 바꾸기
TEMPERATURE = 0.7              # 0이면 매번 같은 답만 나와서 여러 번 읽는 의미가 없음
MIN_READS, MAX_READS = 2, 3    # 2번 읽어 같으면 끝, 다르면 3번째까지 (비용 절약: 적응형 읽기)
READ_MODE = 'sheet'            # 'sheet': 한 장의 칸들을 한 이미지로 모아 한 번에 읽기 (빠름)
READ_STRATEGY = 'safe'         # 'safe': 2번 읽어 다르면 3번째까지 → 같은 칸을 여러 번 읽은 일치율을 확신도로 씀 (보고서 수치는 이걸로)
                               # 'fast': 한 번 읽고 애매한 칸만 한 번 더 (빠르지만 대부분 한 번만 읽어 일치율 정보가 약함)
                               # 'cell' : 칸마다 따로 읽기 (호출이 장당 11~16번으로 많지만 칸끼리 서로 영향이 없음)
READ_WORKERS = 4               # 동시에 읽을 개수 (분당 사용량 제한 에러가 나면 줄이기)
SHEET_TIME_LIMIT = 90          # 답안지 한 장을 읽는 최대 초 (노트북은 정확도 우선. 웹페이지 시연은 5-2에서 30초)
CALL_TIMEOUT = 45              # Gemini 호출 한 번을 기다리는 최대 초
AUTO_PICK_MODEL = True         # 4-3에서 계정에서 되는 Flash 모델들을 실제 답안지로 시험해 가장 빠르고 제대로 읽는 모델을 고름
READ_THINKING_LEVEL = 'MINIMAL'   # 글씨를 옮겨 적는 일이라 생각 단계를 최소로 (지원 안 하는 모델이면 자동으로 끔)
PRICE_PER_1M_TOKENS = None     # 비용 계산용: 쓰는 모델의 100만 토큰당 가격(원). 공식 요금표에서 확인해 입력

# ── 학습 ─────────────────────────────────────────────────────────
N_SAMPLES = 3000               # 시뮬레이션 학습 데이터 개수
EPOCHS, BATCH_SIZE = 60, 64    # 최대 60 에포크, 조기 종료(EarlyStopping)가 더 일찍 멈출 수 있음
RUN_SEEDS = True               # 3-6 여러 시드 반복 (발표 숫자를 범위로 내기)
SEEDS = [42, 1, 2, 3, 4]

### 1-3. 파일 올리기 (정답표, 명렬표, 학생 답안 표)
업로드 창에서 파일을 **한꺼번에** 고르세요. CSV·엑셀 아무거나 되고, 엑셀 한 파일에 시트로 넣어도 됩니다. 열 이름을 보고 **어떤 표인지 알아서 구분**합니다.
- `정답` 열이 있으면 **정답표**, `1번`·`Q1`·`문항1` 같은 열이 있으면 **학생 답안**, `반·번호·이름`이 있으면 **명렬표**
- 열 이름이 조금 달라도 알아듣습니다 (문제번호 → 문항, 점수 → 배점, 성명 → 이름 등)
- 이미 올렸으면 업로드 창을 취소해도 `inputs/data/`의 파일을 그대로 씁니다

In [ ]:
# ── 어떤 파일이든 받아서 알아서 분류하기 ──────────────────────────
TABLE_EXT = ('.csv', '.tsv', '.txt', '.xlsx', '.xls')
ALIASES = {   # 열 이름이 조금 달라도 알아듣기
    '문항': ['문항', '문항번호', '문제', '문제번호', 'question', 'q'],
    '배점': ['배점', '점수', '배점(점)', 'points'],
    '정답': ['정답', '답', '모범답안', 'answer'],
    '허용답안': ['허용답안', '허용', '인정답안', '동의어', 'allowed'],
    '등록오답': ['등록오답', '오답', '오답예시', 'wrong'],
    '반': ['반', '학급', 'class'],
    '번호': ['번호', '출석번호', 'number', 'no'],
    '이름': ['이름', '성명', '학생', 'name'],
    '시험지': ['시험지', '시험지번호', '답안지', 'sheet', 'sheet_id'],
}

def upload_to(folder, label):
    """Colab이면 업로드 창(여러 파일 가능)을 띄운다. 이번에 새로 올린 파일이 있으면 그 파일들만 쓰고
    (예전에 올린 파일은 _이전/ 폴더로 옮김), 아무것도 안 올리고 창을 닫으면 folder에 있던 파일을 그대로 쓴다."""
    os.makedirs(folder, exist_ok=True)
    new = []
    try:
        from google.colab import files
        print(f'▶ {label}')
        new = list(files.upload().items())
    except ImportError:
        pass
    except Exception as e:   # 업로드 창을 닫은 경우 등
        print('업로드를 건너뜀:', type(e).__name__)
    if new:
        old_dir = os.path.join(folder, '_이전')
        os.makedirs(old_dir, exist_ok=True)
        for f in os.listdir(folder):
            if not f.startswith(('.', '_')):
                shutil.move(os.path.join(folder, f), os.path.join(old_dir, f))
        for name, data in new:
            open(os.path.join(folder, name), 'wb').write(data)
    paths = sorted(os.path.join(folder, f) for f in os.listdir(folder) if not f.startswith(('.', '_')))
    print(f'{folder}/ 파일 {len(paths)}개:', [os.path.basename(p) for p in paths])
    return paths

def extract_zip(path, dest):
    """ZIP 풀기 (한글 파일 이름 깨짐 방지: 윈도우 압축(CP949)과 UTF-8 모두 처리)"""
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:   # UTF-8 표시가 없는 이름은 원래 인코딩으로 다시 읽기
                raw = name.encode('cp437', errors='ignore')
                for enc in ('utf-8', 'cp949'):
                    try:
                        name = raw.decode(enc); break
                    except UnicodeDecodeError:
                        pass
            if '__MACOSX' in name or name.endswith('/'):
                continue
            out = os.path.join(dest, *[p for p in name.split('/') if p not in ('', '..')])
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(info) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)

def _decode(raw, label):
    for enc in ('utf-8-sig', 'cp949'):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    raise ValueError(f'{label}: 글자 인코딩을 읽을 수 없습니다 (UTF-8 또는 CP949로 저장하세요)')

def _clean(df):
    df = df.astype(object).where(pd.notna(df), '').astype(str).apply(lambda c: c.str.strip())
    df = df.replace({'nan': '', 'None': '', 'NaN': ''})
    df = df.apply(lambda c: c.str.replace(r'^(\d+)\.0$', r'\1', regex=True))   # 엑셀 숫자 20.0 → 20
    df.columns = [str(c).strip() for c in df.columns]
    rename = {}
    for canon, names in ALIASES.items():
        if canon not in df.columns:
            hit = next((c for c in df.columns if c.lower().replace(' ', '') in names), None)
            if hit:
                rename[hit] = canon
    for c in df.columns:   # 학생 답안 문항 열: 1, Q1, 문항1, 1번 → '1번'
        m = re.fullmatch(r'(?:문항|q)?\s*(\d+)\s*번?', c.lower())
        if m and c not in rename and c not in ALIASES:
            rename[c] = f'{int(m.group(1))}번'
    df = df.rename(columns=rename)
    return df[(df != '').any(axis=1)].reset_index(drop=True)

def read_tables(path):
    """CSV, TSV, TXT, 엑셀(시트 여러 개 포함) → [(이름, 표)]"""
    ext = os.path.splitext(path)[1].lower()
    name = os.path.basename(path)
    if ext == '.xls':
        raise ValueError(f'{name}: 옛날 엑셀(.xls)은 못 읽습니다. 엑셀에서 "다른 이름으로 저장" → .xlsx나 CSV로 저장해 주세요')
    if ext == '.xlsx':
        return [(f'{name}:{s}', _clean(df)) for s, df in pd.read_excel(path, sheet_name=None, dtype=str).items()]
    text = _decode(open(path, 'rb').read(), name)
    sep = '\t' if ext == '.tsv' or text.count('\t') > text.count(',') else ','
    return [(name, _clean(pd.read_csv(io.StringIO(text), sep=sep, dtype=str)))]

def kind_of(df):
    """표의 열을 보고 정답표 / 학생 답안 / 명렬표를 구분"""
    if '정답' in df.columns:
        return '정답표'
    if any(re.fullmatch(r'\d+번', c) for c in df.columns):
        return '학생 답안'
    if {'반', '번호', '이름'} <= set(df.columns):
        return '명렬표'
    return None

TABLES, SEEN, TABLE_NAMES = {}, [], {}
for p in upload_to('inputs/data', '정답표, 명렬표(, 학생 답안) 파일을 고르세요. CSV·엑셀 아무거나, 엑셀 한 파일에 시트 여러 개도 됩니다'):
    if not p.lower().endswith(TABLE_EXT):
        print(f'  표 파일이 아니라 건너뜀: {os.path.basename(p)}')
        continue
    try:
        tables = read_tables(p)
    except Exception as e:
        print(f'  ⚠️ {os.path.basename(p)}을(를) 읽지 못함: {e}')
        continue
    for name, df in tables:
        SEEN.append(f'{name} (열: {list(df.columns)})')
        k = kind_of(df)
        if k is None:
            print(f'  어떤 표인지 알 수 없어 건너뜀: {name} (열: {list(df.columns)})')
        elif k in TABLES:
            print(f'  {k}가 이미 있어 건너뜀: {name}')
        else:
            TABLES[k] = df
            TABLE_NAMES[k] = name
            print(f'  {name} → {k} ({len(df)}행)')
if '정답표' not in TABLES:
    raise ValueError("정답표를 찾지 못했습니다. '문항, 배점, 정답' 열이 있는 CSV나 엑셀을 올려 주세요 "
                     "(예: 시험자료_예시.xlsx).\n읽은 표: " + ('; '.join(SEEN) or '없음 (업로드 창에서 파일을 고르지 않았을 수 있음)'))
KEY_DF, ROSTER_DF, STUDENT_DF = TABLES['정답표'], TABLES.get('명렬표'), TABLES.get('학생 답안')
if ROSTER_DF is None:
    print('⚠️ 명렬표가 없어 학생 식별은 읽은 반·번호·이름을 그대로 씁니다')

### 1-4. 정답표 확인
열이 빠졌거나, 문항·배점이 숫자가 아니거나, 정답이 비었거나, 문항 번호가 겹치면 **몇 행이 문제인지** 알려 주고 멈춥니다. 영어·숫자·화학식 정답도 됩니다.

| 열 | 예 |
|---|---|
| 문항, 배점, 정답 (필수) | 2, 20, 팔만대장경 |
| 허용답안 (`\|`로 여러 개) | 고려대장경\|재조대장경 |
| 등록오답 (비워도 됨) | 초조대장경\|신라대장경 |

In [ ]:
def check(errors, label):
    if errors:
        raise ValueError(f'{label}에 문제가 있습니다:\n  - ' + '\n  - '.join(errors))

key_df = KEY_DF.copy()
check([f"'{c}' 열이 없습니다" for c in ('문항', '배점', '정답') if c not in key_df.columns], '정답표')
for c in ('허용답안', '등록오답', '메모'):
    if c not in key_df.columns:
        key_df[c] = ''
errors = []
for i, r in key_df.iterrows():
    if not r['문항'].isdigit():
        errors.append(f"{i + 2}행: 문항 번호 '{r['문항']}'이(가) 숫자가 아닙니다")
    if not r['배점'].isdigit() or int(r['배점']) <= 0:
        errors.append(f"{i + 2}행: 배점 '{r['배점']}'이(가) 양의 정수가 아닙니다")
    if not r['정답']:
        errors.append(f'{i + 2}행: 정답이 비어 있습니다')
if key_df['문항'].duplicated().any():
    errors.append(f"문항 번호가 겹칩니다: {key_df['문항'][key_df['문항'].duplicated()].tolist()}")
check(errors, '정답표')

_split = lambda s: [a.strip() for a in str(s).split('|') if a.strip()]
ANSWER_KEY = {int(r['문항']): [r['정답']] + _split(r['허용답안']) for _, r in key_df.iterrows()}   # 문항 → 인정하는 답
REGISTERED_WRONG = {int(r['문항']): _split(r['등록오답']) for _, r in key_df.iterrows()}      # 문항 → 확실한 오답
POINTS = {int(r['문항']): int(r['배점']) for _, r in key_df.iterrows()}                       # 문항 → 배점

def normalize(s):
    """띄어쓰기 제거, 가운뎃점 통일, (설정에 따라) 영어 소문자로"""
    s = unicodedata.normalize('NFC', str(s or ''))
    s = re.sub(r'\s+', '', s)
    for ch in '.．ㆍ•∙⋅・':
        s = s.replace(ch, '·')
    return s if CASE_SENSITIVE else s.lower()

def accepted(q):
    return list(dict.fromkeys(normalize(a) for a in ANSWER_KEY[q]))

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

# 한글 자모와 혼동쌍 (받침 ㄴ/ㅇ, ㄴ/ㅁ 포함) + 숫자·영어 혼동
CHO = list('ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ')
JUNG = list('ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ')
JONG = [''] + list('ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ')
CONFUSABLE = [('ㅁ', 'ㅂ'), ('ㅐ', 'ㅔ'), ('ㅗ', 'ㅜ'), ('ㅏ', 'ㅓ'), ('ㅅ', 'ㅈ'), ('ㄴ', 'ㄹ'), ('ㅇ', 'ㅁ'),
              ('ㅓ', 'ㅕ'), ('ㅏ', 'ㅑ'), ('ㅗ', 'ㅛ'), ('ㅜ', 'ㅠ'), ('ㅡ', 'ㅜ'), ('ㄱ', 'ㅋ'), ('ㄷ', 'ㅌ'),
              ('ㄴ', 'ㅇ'), ('ㄴ', 'ㅁ')]
SIMPLE_CONFUSABLE = {'1': ['7', 'l'], '7': ['1'], '3': ['8'], '8': ['3'], '0': ['6', 'o'], '6': ['0'],
                     'o': ['0', 'a'], 'a': ['o'], 'l': ['1', 'i'], 'i': ['l'], 'u': ['v'], 'v': ['u'],
                     'n': ['h'], 'h': ['n'], 'e': ['c'], 'c': ['e']}

def decompose(ch):
    code = ord(ch) - 0xAC00
    if not (0 <= code < 11172):
        return None
    return CHO[code // 588], JUNG[(code % 588) // 28], JONG[code % 28]

def compose(cho, jung, jong=''):
    return chr(0xAC00 + (CHO.index(cho) * 21 + JUNG.index(jung)) * 28 + JONG.index(jong))

def is_confusable(a, b):
    if b in SIMPLE_CONFUSABLE.get(a, []) or a in SIMPLE_CONFUSABLE.get(b, []):
        return True
    da, db = decompose(a), decompose(b)
    if da is None or db is None:
        return False
    diffs = [(x, y) for x, y in zip(da, db) if x != y]
    if len(diffs) != 1:
        return False
    x, y = diffs[0]
    return x == '' or y == '' or (x, y) in CONFUSABLE or (y, x) in CONFUSABLE

display(key_df)
print('인정하는 답 (띄어쓰기·기호·대소문자 정리 후):', {q: accepted(q) for q in ANSWER_KEY})

### 1-5. 특징 만들기
읽기 결과를 신경망이 이해하는 **숫자 8개**로 바꿉니다: 정답읽힘비율, 읽기일치율, 글자유사도, 길이차이, 혼동쌍만다름, 순서만다름, 빈칸, 지저분함. 아래 3행짜리 작은 표로 숫자가 맞는지 먼저 눈으로 확인하세요.

In [ ]:
FEATURES = ['정답읽힘비율', '읽기일치율', '글자유사도', '길이차이', '혼동쌍만다름', '순서만다름', '빈칸', '지저분함']

def top_read(texts):
    """가장 많이 나온 읽기 결과와 그 비율"""
    top, n = Counter(normalize(t) for t in texts).most_common(1)[0]
    return top, n / len(texts)

def make_features(texts, q, messy=False):
    """AI가 같은 답안을 여러 번 읽은 결과(texts) → 신경망에 넣을 숫자 8개"""
    texts = [normalize(t) for t in texts]
    ok = set(accepted(q))
    top, agree = top_read(texts)                                   # 가장 많이 나온 읽기 결과
    best = min(accepted(q), key=lambda a: edit_distance(top, a))   # 가장 가까운 허용 답안
    d = edit_distance(top, best)
    mism = [(x, y) for x, y in zip(top, best) if x != y] if len(top) == len(best) else []
    registered = top in {normalize(w) for w in REGISTERED_WRONG.get(q, [])}   # 교사가 등록한 오답 (예: 기역 → 기억)
    return {
        '정답읽힘비율': sum(t in ok for t in texts) / len(texts),   # 허용 답안으로 읽힌 비율
        '읽기일치율': agree,                                        # 읽을 때마다 같은 결과가 나온 비율
        '글자유사도': 1 - d / max(len(top), len(best), 1),          # 1이면 허용 답안과 똑같음
        '길이차이': min(abs(len(top) - len(best)), 3) / 3,
        # 등록한 오답은 혼동쌍·순서 규칙에 걸리지 않게 해서 그대로 오답이 되게 함
        '혼동쌍만다름': float(not registered and bool(mism) and all(is_confusable(x, y) for x, y in mism)),  # 예: 훈민정읍
        '순서만다름': float(not registered and top != best and sorted(top) == sorted(best)),               # 예: 이신순
        '빈칸': float(top == ''),
        '지저분함': float(bool(messy)),                             # 지운 자국, 답 두 개 등
    }

def near_miss(texts, q):
    """한 글자 차이이거나 정답 뒤에 말이 붙은 답인지 (교사가 등록한 오답은 제외: '평양'과 '한양').
    쉼표로 답을 두 개 이상 쓴 경우는 'two'"""
    top, _ = top_read(texts)
    if ',' in top or '，' in top:
        return 'two'
    wrong = {normalize(w) for w in REGISTERED_WRONG.get(q, [])}
    if not top or top in accepted(q) or top in wrong:
        return False
    return any(edit_distance(top, a) == 1 or top.startswith(a) for a in accepted(q))

# 작은 표로 먼저 확인: 정답표의 첫 문항으로 3가지 상황을 만들어 특징을 계산합니다.
q0 = list(ANSWER_KEY)[0]
a0 = ANSWER_KEY[q0][0]
typo0 = a0[:-1] + ('읍' if a0.endswith('음') else '?')        # 마지막 글자만 다르게 읽힌 경우
wrong0 = (REGISTERED_WRONG[q0] or ['전혀다른답'])[0]
small = [
    (f'5번 모두 "{a0}"로 읽음', q0, [a0] * 5),
    (f'3번 "{a0}", 2번 "{typo0}"', q0, [a0] * 3 + [typo0] * 2),
    (f'5번 모두 "{wrong0}"로 읽음', q0, [wrong0] * 5),
]
pd.DataFrame([{'상황': s, **make_features(t, q, False)} for s, q, t in small]).round(2)

---
## 2단계. 학습 데이터 만들기
**학생이 쓴 답**(정답 45%, 등록오답 20%, 혼동쌍 오타 12%, 일반 오타 8%, 순서 바뀜 8%, 빈칸 7%)과 **읽기 실수**(글씨가 나쁠수록 혼동쌍으로 잘못 읽기, 정답으로 고쳐 읽기)를 흉내 내서 3,000개를 만듭니다. 읽는 방식도 4단계와 같은 적응형(2~3번)입니다. 그 문항으로 만들 수 없는 유형은 건너뜁니다.

> **주의!** 이 데이터는 **정답 열(정답여부)이 맨 앞**에 있습니다.

In [ ]:
def confusion_variants(ch):
    """글자 하나를 혼동쌍으로 잘못 읽었을 때 나올 수 있는 글자들"""
    if ch in SIMPLE_CONFUSABLE:
        return list(SIMPLE_CONFUSABLE[ch])
    d = decompose(ch)
    if d is None:
        return []
    cho, jung, jong = d
    out = set()
    for a, b in CONFUSABLE:
        for x, y in ((a, b), (b, a)):
            if cho == x and y in CHO: out.add(compose(y, jung, jong))
            if jung == x and y in JUNG: out.add(compose(cho, y, jong))
            if jong == x and y in JONG: out.add(compose(cho, jung, y))
    if jong:
        out.add(compose(cho, jung, ''))
    else:
        out.update(compose(cho, jung, j) for j in ('ㄴ', 'ㅇ'))
    out.discard(ch)
    return sorted(out)

# 학생 답 유형별로 답 하나를 만드는 함수 → (답, 정답여부). 그 문항으로 만들 수 없으면 IndexError
def make_correct(q):
    return random.choice(ANSWER_KEY[q]), 1
def make_wrong(q):              # 그럴듯한 오답: 정답표의 등록오답 열에서만 가져옴
    return random.choice(REGISTERED_WRONG[q]), 0
def make_confusion_typo(q):     # 훈민정읍, DNA → DNR 같은 혼동쌍 오타
    base = random.choice(accepted(q))
    i = random.choice([i for i, c in enumerate(base) if confusion_variants(c)])
    return base[:i] + random.choice(confusion_variants(base[i])) + base[i + 1:], 0
def make_typo(q):               # 이순산, photosynthasis 같은 일반 오타
    base = random.choice(accepted(q))
    i = random.choice([i for i, c in enumerate(base) if decompose(c) or c.isascii() and c.isalpha()])
    c = compose(random.choice(CHO), random.choice(JUNG)) if decompose(base[i]) \
        else random.choice([x for x in 'abcdefghijklmnopqrstuvwxyz' if x != base[i]])
    return base[:i] + c + base[i + 1:], 0
def make_shuffle(q):            # 이신순
    base = random.choice([a for a in accepted(q) if len(set(a)) > 1])
    t = base
    while t == base:
        t = ''.join(random.sample(base, len(base)))
    return t, 0
def make_blank(q):
    return '', 0

KINDS = {'정답': (make_correct, 0.45), '그럴듯한오답': (make_wrong, 0.20), '혼동쌍오타': (make_confusion_typo, 0.12),
         '일반오타': (make_typo, 0.08), '순서바뀜': (make_shuffle, 0.08), '빈칸': (make_blank, 0.07)}

def student_answer(q):
    while True:
        kind = random.choices(list(KINDS), weights=[w for _, w in KINDS.values()])[0]
        try:
            text, label = KINDS[kind][0](q)
            break
        except IndexError:   # 이 문항으로는 만들 수 없는 유형 (영어 정답의 한글 혼동쌍, 등록오답 없음 등) → 다시 고름
            continue
    if normalize(text) in accepted(q):   # 우연히 허용 답안이 되면 정답으로
        label = 1
    return text, label, kind

def simulate_reads(text, q):
    """같은 답안을 읽는 것을 흉내 낸다 (4단계와 같은 적응형: 2번 읽어 같으면 끝, 다르면 3번째까지).
    글씨가 나쁠수록 혼동쌍으로 잘못 읽고, 정답과 한 글자 차이면 정답으로 '고쳐 읽는' 실수도 한다."""
    text = normalize(text)
    bad = np.random.beta(2, 5)                         # 글씨 상태: 0 깔끔 ~ 1 악필
    messy = random.random() < 0.06                     # 지운 자국 등
    err = 0.01 + 0.25 * bad + (0.2 if messy else 0)    # 글자 하나를 잘못 읽을 확률
    near = [a for a in accepted(q) if len(a) == len(text) and edit_distance(text, a) == 1]
    reads = []
    for k in range(MAX_READS):
        if near and random.random() < 0.1 + 0.3 * bad:
            reads.append(random.choice(near))
        else:
            reads.append(''.join(random.choice(confusion_variants(c)) if random.random() < err and confusion_variants(c) else c
                                 for c in text))
        if k + 1 >= MIN_READS and len(set(reads)) == 1:
            break
    return reads, messy

def make_dataset(n_samples, seed):
    """시뮬레이션 데이터 n개 (시드를 바꾸면 다른 데이터)"""
    random.seed(seed); np.random.seed(seed)
    rows = []
    for _ in range(n_samples):
        q = random.choice(list(ANSWER_KEY))
        text, label, kind = student_answer(q)
        reads, messy = simulate_reads(text, q)
        rows.append({'정답여부': label, **make_features(reads, q, messy), '문항': q, '학생답': normalize(text),
                     '유형': kind, '근처': near_miss(reads, q), '읽기': ' / '.join(t or '(빈칸)' for t in reads)})
    return pd.DataFrame(rows)

sim = make_dataset(N_SAMPLES, SEED)
sim[['정답여부'] + FEATURES].to_csv('answer_data.csv', index=False, encoding='utf-8-sig')   # 정답 열이 맨 앞
print(f'{N_SAMPLES}개 생성 → answer_data.csv')
print(sim['유형'].value_counts().to_string())
sim.head(8)

### 2-1. 데이터 불러오기
맨 앞 열이 정답이므로 `iloc[:, 1:]`이 특징(X), `iloc[:, 0]`이 정답(y)입니다.

In [ ]:
data = pd.read_csv('answer_data.csv', encoding='utf-8-sig')
X = data.iloc[:, 1:].values   # 맨 앞(정답 열)을 뺀 나머지 = 특징 8개
y = data.iloc[:, 0].values    # 맨 앞 열 = 정답 여부 (1 정답, 0 오답)
print('X:', X.shape, ' y:', y.shape)

X_train, X_test, y_train, y_test, i_train, i_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y)
print('학습용', len(y_train), '/ 시험용', len(y_test))

---
## 3단계. 신경망 학습
### 3-1. 모델 만들기와 학습
정답(1)/오답(0) **이진분류**이고, `sigmoid` 출력이 곧 **P(정답)**입니다. `batch_size=64`와 **조기 종료**(검증 loss가 10 에포크 동안 안 줄면 멈추고 가장 좋던 가중치로 되돌림)로 빠르게 학습합니다.

In [ ]:
def build_model():
    model = Sequential()
    model.add(Dense(10, input_dim=8, activation='relu'))   # [작성] X.shape의 두 번째 숫자 = 특징 개수 8
    model.add(Dense(1, activation='sigmoid'))               # 출력 1개 = P(정답), 0~1 사이 확률
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def train(model, X, y):
    """검증 loss가 10 에포크 동안 안 줄면 멈추고, 가장 좋았던 가중치로 되돌림 (속도 + 과적합 방지)"""
    return model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=0,
                     callbacks=[EarlyStopping(patience=10, min_delta=0.001, restore_best_weights=True)])

t0 = time.time()
model = build_model()
history = train(model, X_train, y_train)
h = pd.DataFrame(history.history)
best = int(h['val_loss'].idxmin())
print(f"{len(h)} 에포크에서 멈춤 ({time.time() - t0:.0f}초) · 가장 좋은 {best + 1}번째: 검증 loss {h['val_loss'][best]:.3f}, "
      f"검증 정확도 {h['val_accuracy'][best]:.1%}")

### 3-2. 학습 곡선
학습셋과 검증셋 선이 함께 내려가고 차이가 작으면 과적합 없이 학습된 것입니다.

In [ ]:
curve = h.reset_index(names='epoch').assign(epoch=lambda d: d['epoch'] + 1)
long = pd.concat([
    curve.melt('epoch', ['loss', 'val_loss'], 'set', 'value').assign(metric='손실(loss)'),
    curve.melt('epoch', ['accuracy', 'val_accuracy'], 'set', 'value').assign(metric='정확도(accuracy)')])
long['set'] = long['set'].map(lambda s: '검증셋' if s.startswith('val') else '학습셋')
g = sns.relplot(data=long, x='epoch', y='value', hue='set', col='metric', kind='line',
                facet_kws={'sharey': False}, height=3.8, aspect=1.4, palette={'학습셋': '#1f77b4', '검증셋': '#d62728'})
for ax in g.axes.flat:
    ax.axvline(best + 1, color='gray', ls='--', lw=1)
g.set_titles('{col_name}'); g.set_axis_labels('epoch', '')
g.figure.suptitle('학습 곡선 (점선 = 검증 loss 최저, 이 시점의 가중치를 씀)', y=1.04)
g.savefig('results/1_learning_curve.png', dpi=150, bbox_inches='tight'); plt.show()

### 3-3. 판정 규칙과 시험용 데이터 확인
`rule_check()`가 교사 확인 규칙을 먼저 적용하고, 나머지를 `decide()`가 기준값으로 판정합니다. 오른쪽 그래프에서 **그럴듯한 오답은 오답**, **혼동쌍·순서 바뀜·오타는 교사 확인**으로 가는지 보세요.

In [ ]:
def rule_check(f, outside=False, near=False, mixed=None, near_rule=None):
    """확률과 상관없이 교사 확인으로 보내는 규칙 (정리 문서 4절 판정 순서). 걸리면 사유, 아니면 None"""
    mixed = MIXED_READS_RULE if mixed is None else mixed
    near_rule = NEAR_MISS_TO_REVIEW if near_rule is None else near_rule
    r = f['정답읽힘비율']
    if outside:
        return '칸 밖에 쓴 글씨가 있음'
    if near == 'two':
        return '답을 두 개 이상 씀 (어느 답을 채점할지 교사가 결정)'
    if f['지저분함']:
        return '칸이 지저분함 (지운 자국, 답 두 개 등)'
    if f['순서만다름']:
        return '글자는 같고 순서만 다름 (어순 정책은 교사가 결정)'
    if f['혼동쌍만다름']:
        return '혼동쌍만 다름 (학생 오타인지 글씨 문제인지 구분 불가)'
    if (mixed == 'strict' and 0 < r < 1) or (mixed == 'lenient' and 0 < r < 0.8):
        return '읽기 결과가 엇갈림 (어느 읽기를 믿느냐에 따라 판정이 바뀜)'
    if near_rule and near:
        return '한 글자 차이이거나 정답 뒤에 말이 붙음 (오타 정책은 교사가 결정)'
    return None

def decide(p, f, outside=False, near=False):
    """신경망 확률(p) + 교사 정책 규칙 → (판정, 사유)"""
    reason = rule_check(f, outside, near)
    if reason:
        return '교사 확인', reason
    if p >= T_HIGH:
        return '정답', f'P(정답) {p:.0%} ≥ {T_HIGH:.0%}'
    if p <= T_LOW:
        return '오답', f'P(정답) {p:.0%} ≤ {T_LOW:.0%}'
    return '교사 확인', f'P(정답) {p:.0%}: 애매한 구간'

def review_mask(df, mixed=None, near_rule=None):
    """규칙에 걸려 교사 확인으로 가는 행 (True)"""
    return np.array([rule_check(f, False, n, mixed, near_rule) is not None
                     for (_, f), n in zip(df[FEATURES].iterrows(), df['근처'])])

def judge_array(P, mask, t_high, t_low):
    return np.where(mask, '교사 확인', np.where(P >= t_high, '정답', np.where(P <= t_low, '오답', '교사 확인')))

def auto_stats(dec, y):
    """→ 자동 처리 비율, 자동 처리 수, 오답을 정답 처리, 정답을 오답 처리"""
    auto = dec != '교사 확인'
    return auto.mean(), int(auto.sum()), int(((dec == '정답') & (y == 0)).sum()), int(((dec == '오답') & (y == 1)).sum())

loss, acc = model.evaluate(X_test, y_test, verbose=0)
test = sim.iloc[i_test].copy()
P_test = model.predict(X_test, verbose=0).ravel()
test['P(정답)'] = P_test
mask_test = review_mask(test)
test['판정'] = judge_array(P_test, mask_test, T_HIGH, T_LOW)
test['실제'] = test['정답여부'].map({1: '실제 정답', 0: '실제 오답'})
print(f'시험용 {len(test)}개 · 정확도(0.5 기준) {acc:.1%}')

by_kind = pd.crosstab(test['유형'], test['판정'])
by_kind['평균 P(정답)'] = test.groupby('유형')['P(정답)'].mean().round(2)
display(by_kind)

def threshold_lines(ax, axis='x'):
    for t in (T_LOW, T_HIGH):
        (ax.axvline if axis == 'x' else ax.axhline)(t, color='k', ls='--', lw=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.histplot(data=test, x='P(정답)', hue='실제', bins=np.linspace(0, 1, 21), multiple='stack',
             palette={'실제 정답': '#1f77b4', '실제 오답': '#d62728'}, ax=axes[0])
threshold_lines(axes[0]); axes[0].set_title(f'시험용 데이터 {len(test)}개의 P(정답) 분포 (점선 = 기준값)')
sns.countplot(data=test, y='유형', hue='판정', hue_order=JUDGE_ORDER, palette=JUDGE_COLOR, ax=axes[1],
              order=test['유형'].value_counts().index)
axes[1].set_title('답 유형별 판정 (혼동쌍·순서 바뀜은 교사 확인으로 가야 함)'); axes[1].set_xlabel('답안 수')
plt.tight_layout(); plt.savefig('results/2_test_eval.png', dpi=150, bbox_inches='tight'); plt.show()

### 3-4. 기준값 찾기와 판정 규칙의 득실
허용할 오류(`TARGET_FALSE_CORRECT` 등)를 먼저 정하고, **검증용 데이터**에서 그것을 만족하는 기준값을 찾습니다(기본값보다 느슨하게는 안 내림). 아래 표와 그래프는 엇갈린 읽기를 얼마나 엄격하게 볼지에 따른 **자동 처리 비율과 오류의 득실**, 그리고 **신경망 vs 단순 규칙** 비교입니다.

In [ ]:
# 허용할 오류를 먼저 정하고, 그것을 만족하는 기준값을 검증용 데이터에서 거꾸로 찾는다 (정리 문서 5절)
TARGET_FALSE_CORRECT = 0.01    # 자동 정답 중 실제로는 오답인 비율 상한 (발견하기 어려운 오류라 엄격하게)
TARGET_FALSE_WRONG = 0.02      # 자동 오답 중 실제로는 정답인 비율 상한
USE_FOUND_THRESHOLDS = True    # True면 찾은 값을 4단계 채점에 씀

val = make_dataset(1500, SEED + 1000)          # 학습·시험과 겹치지 않는 검증용 데이터
P_val = model.predict(val[FEATURES].values, verbose=0).ravel()
y_val = val['정답여부'].values
mask_val = review_mask(val)

def find_thresholds(P, y, mask):
    """T_HIGH: 목표를 만족하는 가장 낮은 값, T_LOW: 목표를 만족하는 가장 높은 값.
    단, 기본값(0.8, 0.2)보다 느슨하게는 내리지 않는다 (데이터는 기준을 더 엄격하게만 바꿀 수 있음)"""
    t_high = next((t for t in np.round(np.arange(0.80, 1.00, 0.01), 2)
                   if (sel := ~mask & (P >= t)).sum() > 0 and (y[sel] == 0).mean() <= TARGET_FALSE_CORRECT), None)
    t_low = next((t for t in np.round(np.arange(0.20, 0.0, -0.01), 2)
                  if (sel := ~mask & (P <= t)).sum() > 0 and (y[sel] == 1).mean() <= TARGET_FALSE_WRONG), None)
    return t_high, t_low

found_high, found_low = find_thresholds(P_val, y_val, mask_val)
print(f'검증용 {len(val)}개에서 찾은 기준값: T_HIGH = {found_high}, T_LOW = {found_low}  (지금 설정 {T_HIGH}, {T_LOW})')
if found_high is None or found_low is None:
    grid = [(t, (y_val[s] == 0).mean(), s.sum()) for t in np.round(np.arange(0.80, 1.00, 0.01), 2)
            if (s := ~mask_val & (P_val >= t)).sum() > 0]
    t_b, e_b, n_b = min(grid, key=lambda g: g[1]) if grid else (None, float('nan'), 0)
    print(f'⚠️ 자동 정답 목표({TARGET_FALSE_CORRECT:.0%})를 만족하는 기준값이 없어 지금 설정을 그대로 씁니다. '
          f'가장 나은 T_HIGH {t_b}에서도 자동 정답 {n_b}개 중 오류 {e_b:.1%}')
    print('   → 읽기가 모두 일치해도 틀리는 경우(정답으로 고쳐 읽기)가 있다는 뜻: MAX_READS를 늘리거나 목표를 조정하세요')
for name, t, sel, bad in (('자동 정답', found_high, lambda t: ~mask_val & (P_val >= t), 0),
                          ('자동 오답', found_low, lambda t: ~mask_val & (P_val <= t), 1)):
    if t is not None:
        s_ = sel(t)
        print(f'  검증용에서 {name} {s_.sum()}개 중 틀린 판정 {(y_val[s_] == bad).sum()}개')
if USE_FOUND_THRESHOLDS and found_high is not None and found_low is not None:
    T_HIGH, T_LOW = float(found_high), float(found_low)
    print(f'→ 4단계 채점에 T_HIGH = {T_HIGH}, T_LOW = {T_LOW}를 씁니다')

# 판정 규칙에 따른 득실 + 신경망과 단순 규칙 비교 (시험용 데이터, "안전에는 대가가 있다")
#   단순 규칙 = 신경망 대신 '정답읽힘비율'을 그대로 확률로 쓰고, 나머지 규칙과 기준값은 똑같이 적용
rows = []
P_rule = test['정답읽힘비율'].values.astype(float)
for name, mixed, near_rule in (('원본 (엇갈림·한 글자 규칙 없음)', 'off', False), ('한 글자 차이 규칙만', 'off', True),
                               ('80% 미만으로 엇갈리면 교사 확인', 'lenient', True), ('하나라도 엇갈리면 교사 확인', 'strict', True)):
    mask = review_mask(test, mixed, near_rule)
    row = {'판정 규칙': name + (' ← 현재' if (mixed, near_rule) == (MIXED_READS_RULE, NEAR_MISS_TO_REVIEW) else '')}
    for label, P in (('신경망', P_test), ('단순 규칙', P_rule)):
        rate, n_auto, fa, fr = auto_stats(judge_array(P, mask, T_HIGH, T_LOW), y_test)
        row[f'{label} 자동 처리'] = f'{rate:.1%}'
        row[f'{label} 오류'] = f'{fa + fr}건 ({(fa + fr) / max(n_auto, 1):.2%})'
    rows.append(row)
policy_table = pd.DataFrame(rows)
display(policy_table)
policy_table.to_csv('results/판정규칙_비교.csv', index=False, encoding='utf-8-sig')

# 그래프: 판정 규칙별 자동 처리 비율과 오류 (신경망 vs 단순 규칙)
plot_rows = []
for name, mixed, near_rule in (('원본', 'off', False), ('한 글자 규칙만', 'off', True),
                               ('80% 미만 엇갈림', 'lenient', True), ('하나라도 엇갈림', 'strict', True)):
    mask = review_mask(test, mixed, near_rule)
    for label, P in (('신경망', P_test), ('단순 규칙', P_rule)):
        rate, n_auto, fa, fr = auto_stats(judge_array(P, mask, T_HIGH, T_LOW), y_test)
        plot_rows.append({'규칙': name, '방식': label, '자동 처리 비율(%)': rate * 100, '자동 처리 중 오류(건)': fa + fr})
plot_df = pd.DataFrame(plot_rows)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=plot_df, x='규칙', y='자동 처리 비율(%)', hue='방식', ax=axes[0])
sns.barplot(data=plot_df, x='규칙', y='자동 처리 중 오류(건)', hue='방식', ax=axes[1])
for ax in axes:
    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.0f', fontsize=9)
axes[0].set_title('엄격할수록 교사 확인이 늘고 (자동 처리 ↓)'); axes[1].set_title('대신 자동 처리 중 오류가 줄어든다')
plt.tight_layout(); plt.savefig('results/3_policy_tradeoff.png', dpi=150, bbox_inches='tight'); plt.show()

### 3-5. 보정: 신경망이 말한 확률을 믿어도 되는가
점이 대각선에 가까울수록, **ECE**가 0에 가까울수록 확률을 믿을 만합니다. 신경망과 단순 규칙을 같이 그렸습니다.

In [ ]:
def calibration(P, y, bins=10):
    """확률 구간별 (평균 예측 확률, 실제 정답률, 개수)와 ECE"""
    idx = np.clip(np.digitize(P, np.linspace(0, 1, bins + 1)) - 1, 0, bins - 1)
    table = pd.DataFrame({'구간': idx, 'P': P, 'y': y}).groupby('구간').agg(
        말한확률=('P', 'mean'), 실제정답률=('y', 'mean'), 답안수=('y', 'size')).reset_index()
    ece = float((table['답안수'] / len(P) * (table['말한확률'] - table['실제정답률']).abs()).sum())
    return table, ece

cal_nn, ece = calibration(P_test, y_test)
cal_rule, ece_rule = calibration(test['정답읽힘비율'].values.astype(float), y_test)
print(f'ECE (0에 가까울수록 확률을 믿을 만함): 신경망 {ece:.3f} · 단순 규칙 {ece_rule:.3f}')

cal = pd.concat([cal_nn.assign(방식=f'신경망 (ECE {ece:.3f})'), cal_rule.assign(방식=f'단순 규칙 (ECE {ece_rule:.3f})')])
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], ls='--', color='gray', label='완벽한 보정')
sns.lineplot(data=cal, x='말한확률', y='실제정답률', hue='방식', marker='o', ax=ax)
sns.scatterplot(data=cal, x='말한확률', y='실제정답률', hue='방식', size='답안수', sizes=(30, 300), legend=False, ax=ax)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='말한 P(정답)', ylabel='실제 정답 비율', title='신뢰도 곡선 (점 크기 = 답안 수)')
plt.savefig('results/4_reliability.png', dpi=150, bbox_inches='tight'); plt.show()

### 3-6. 여러 시드로 반복
발표에는 한 번 실행한 값이 아니라 **이 범위**를 쓰세요. 빨리 넘어가려면 1-2에서 `RUN_SEEDS = False`.

In [ ]:
def run_seed(s):
    d = make_dataset(N_SAMPLES, s)
    Xs, ys = d[FEATURES].values, d['정답여부'].values
    Xtr, Xte, ytr, yte, _, ite = train_test_split(Xs, ys, np.arange(len(ys)), test_size=0.2, random_state=s, stratify=ys)
    tf.random.set_seed(s)
    m = build_model()
    train(m, Xtr, ytr)
    P = m.predict(Xte, verbose=0).ravel()
    rate, n_auto, fa, fr = auto_stats(judge_array(P, review_mask(d.iloc[ite]), T_HIGH, T_LOW), yte)
    return {'시드': str(s), '정확도(%)': ((P >= 0.5) == yte).mean() * 100, 'ECE': calibration(P, yte)[1],
            '자동 처리 비율(%)': rate * 100, '자동 처리 중 오류(건)': fa + fr}

if RUN_SEEDS:
    t0 = time.time()
    seed_df = pd.DataFrame([run_seed(s) for s in SEEDS])
    print(f'시드 {len(SEEDS)}개 ({time.time() - t0:.0f}초)')
    display(seed_df.round(3))
    for col in seed_df.columns[1:]:
        print(f'{col}: {seed_df[col].min():.3g} ~ {seed_df[col].max():.3g}')
    seed_df.to_csv('results/seed_results.csv', index=False, encoding='utf-8-sig')
    long = seed_df.melt('시드', var_name='지표', value_name='값')
    g = sns.catplot(data=long, x='시드', y='값', col='지표', kind='bar', sharey=False, height=3, aspect=0.9,
                    color='#4c72b0')
    g.set_titles('{col_name}'); g.figure.suptitle('시드별 결과 (발표에는 이 범위를 씀)', y=1.05)
    g.savefig('results/5_seeds.png', dpi=150, bbox_inches='tight'); plt.show()

### 3-7. 시연 서버로 신경망 내보내기
이 노트북에서 학습한 신경망을 **`grader.json`**(숫자 101개)으로 저장하고 내려받습니다. 시연 웹페이지의 **"정답표·명렬표·신경망 바꾸기"**에 정답표와 함께 올리면, 서버가 **이 노트북의 신경망으로** 채점합니다.
- grader.json을 안 올려도, 시연 서버는 새 정답표를 받으면 **같은 방식(같은 시뮬레이션, 같은 신경망 구조)으로 그 자리에서 다시 학습**합니다. 이 파일은 "노트북에서 검증한 바로 그 모델"을 쓰고 싶을 때 올리면 됩니다.
- 서버는 TensorFlow 없이 numpy로 같은 계산을 하므로, 아래에서 Keras 결과와 차이가 거의 0인지 확인합니다.

In [ ]:
# 학습한 신경망을 시연 서버(API)용 파일 grader.json으로 내보내기
#   시연 웹페이지의 '정답표·명렬표·신경망 바꾸기'에 정답표와 함께 올리면, 이 노트북에서 학습한 신경망으로 채점합니다.
import hashlib
from datetime import datetime, timezone

(W1, b1), (W2, b2) = model.layers[0].get_weights(), model.layers[1].get_weights()
fingerprint = hashlib.sha1(repr(sorted((q, ANSWER_KEY[q][0], tuple(ANSWER_KEY[q][1:]), tuple(REGISTERED_WRONG[q]))
                                       for q in ANSWER_KEY)).encode()).hexdigest()[:12]   # 서버와 같은 정답표 지문

# numpy로 같은 계산을 해서 Keras 결과와 같은지 확인 (서버는 TensorFlow 없이 numpy로 계산함)
sample_X = X_test[:200]
p_np = 1 / (1 + np.exp(-(np.maximum(sample_X @ W1 + b1, 0) @ W2 + b2))).ravel()
diff = float(np.abs(p_np - model.predict(sample_X, verbose=0).ravel()).max())

export = {
    'features': FEATURES,
    'layers': [{'activation': 'relu', 'kernel': W1.tolist(), 'bias': b1.tolist()},
               {'activation': 'sigmoid', 'kernel': W2.tolist(), 'bias': b2.tolist()}],
    'info': {'source': 'notebook', 'answer_key': fingerprint, 'samples': N_SAMPLES, 'seed': SEED,
             'epochs': len(h), 'val_accuracy': round(float(h['val_accuracy'][best]), 4),
             'reads': f'adaptive {MIN_READS}~{MAX_READS}', 'numpy_vs_keras_max_diff': diff,
             'trained_at': datetime.now(timezone.utc).isoformat(timespec='seconds')},
}
json.dump(export, open('results/grader.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'results/grader.json 저장 (정답표 지문 {fingerprint}, numpy와 Keras 차이 {diff:.1e})')
try:
    from google.colab import files
    files.download('results/grader.json')
except Exception:
    pass

---
## 4단계. 채점
### 4-1. 답안지 올리기
사진(jpg, png, **heic**), **여러 쪽 PDF**(쪽마다 답안지 한 장), **ZIP**을 한꺼번에 고르세요. 파일 이름이 시험지 이름이 됩니다. 답안지 이미지 없이 1-3에서 학생 답안 표를 올렸으면 표로 채점합니다.

찍는 팁: 평평한 곳에 놓고, **네 모서리 검은 사각형이 사진 끝에서 조금 떨어지게**, 그림자 없이. 스캔은 **컬러나 회색조**로(흑백 모드는 연한 글씨·지운 자국을 날려 버림).

In [ ]:
IMAGE_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif', '.bmp', '.tif', '.tiff')

def open_image(path):
    """어떤 이미지든 RGB로 열기 (아이폰 HEIC 포함, 사진 회전 정보 반영)"""
    if path.lower().endswith(('.heic', '.heif')):
        import pillow_heif
        pillow_heif.register_heif_opener()
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def expand(paths, out_dir='work/pages'):
    """사진, PDF(여러 쪽), ZIP(사진·PDF 묶음) → 답안지 한 장씩 PNG로 [(시험지, 경로)]"""
    pages = []
    for p in paths:
        stem, ext = os.path.splitext(os.path.basename(p))
        ext = ext.lower()
        if ext == '.zip':
            sub = os.path.join('work', '_zip', stem)
            extract_zip(p, sub)
            inner = sorted(os.path.join(r, f) for r, _, fs in os.walk(sub) for f in fs
                           if not f.startswith('.') and '__MACOSX' not in r)
            pages += expand(inner, out_dir)
        elif ext == '.pdf':
            import fitz   # PyMuPDF
            with fitz.open(p) as doc:
                for i, page in enumerate(doc):
                    out = os.path.join(out_dir, f'{stem}_p{i + 1:02d}.png')
                    page.get_pixmap(dpi=200).save(out)
                    pages.append((f'{stem}_p{i + 1:02d}' if len(doc) > 1 else stem, out))
        elif ext in IMAGE_EXT:
            out = os.path.join(out_dir, f'{stem}.png')
            open_image(p).save(out)
            pages.append((stem, out))
        else:
            print(f'  답안지 형식이 아니라 건너뜀: {os.path.basename(p)}')
    return pages

PAGES = expand(upload_to('inputs/sheets', '답안지 사진·스캔을 고르세요 (jpg, png, heic, pdf 여러 쪽, zip 모두 가능)'))
seen = Counter()
for i, (sid, path) in enumerate(PAGES):   # 이름이 겹치면 뒤에 번호를 붙임
    seen[sid] += 1
    if seen[sid] > 1:
        PAGES[i] = (f'{sid}_{seen[sid]}', path)
IMAGE_MODE = len(PAGES) > 0
if IMAGE_MODE:
    print(f'답안지 {len(PAGES)}장:', [s for s, _ in PAGES])
elif STUDENT_DF is not None:
    print('답안지 이미지가 없어, 1-3에서 올린 학생 답안 표로 채점합니다 (4-2, 4-3은 건너뜀)')
else:
    raise ValueError('답안지 이미지(사진·PDF·ZIP)나 학생 답안 표 중 하나는 있어야 합니다.')

# 명렬표 확인
ROSTER = None
if ROSTER_DF is not None:
    bad = [f"{i + 2}행 ({r['반']}반 {r['번호']}번)" for i, r in ROSTER_DF.iterrows()
           if not (r['반'].isdigit() and r['번호'].isdigit())]
    check([f'반·번호가 숫자가 아님: {b}' for b in bad], '명렬표')
    ROSTER = {(int(r['반']), int(r['번호'])): r['이름'] for _, r in ROSTER_DF.iterrows()}
    print(f'명렬표 {len(ROSTER)}명')

### 4-2. 정렬과 칸 자르기
네 모서리 표시로 사진을 반듯하게 폅니다(그림자에 묻히거나 사진 끝에 잘린 모서리 하나는 나머지 셋으로 추정). 반·번호·이름 칸과 정답 칸을 자르고, 칸 밖에 쓴 글씨가 있으면 ✏️로 표시해 넓게 잘라 둡니다. 미리보기에서 상자가 어긋나면 좌표를 조정하세요.

In [ ]:
# 시험지 PDF에서 잰 위치 (페이지 너비/높이 대비 비율)
CORNERS = np.float32([[0.05, 0.0354], [0.95, 0.0354], [0.05, 0.9646], [0.95, 0.9646]])  # 네 모서리 검은 사각형 중심
ID_BOX = (0.430, 0.160, 0.810, 0.210)                                                   # '( )반 ( )번 이름( )'
SCORE_BOX = (0.762, 0.101, 0.886, 0.155)                                                # 점수 칸
ANSWER_BOXES = {1: (0.190, 0.365, 0.667, 0.419), 2: (0.190, 0.484, 0.667, 0.538), 3: (0.190, 0.603, 0.667, 0.657),
                4: (0.190, 0.723, 0.667, 0.776), 5: (0.190, 0.842, 0.667, 0.896)}
AROUND_BOXES = {1: (0.175, 0.357, 0.950, 0.431), 2: (0.175, 0.476, 0.950, 0.550), 3: (0.175, 0.596, 0.950, 0.669),
                4: (0.175, 0.715, 0.950, 0.788), 5: (0.175, 0.834, 0.950, 0.938)}   # 칸 주변의 원래 빈 영역
PAGE_W, PAGE_H, INSET = 1240, 1754, 0.006
OUTSIDE_MIN_INK = 50   # 칸 밖 잉크 픽셀이 이보다 많으면 '칸 밖 글씨'
PX = lambda v: int(v * PAGE_W)   # 비율 → 픽셀 (학습 데이터 X와 이름이 겹치지 않게 PX, PY)
PY = lambda v: int(v * PAGE_H)

def find_corners(gray):
    """네 모서리 검은 사각형 중심 [좌상, 우상, 좌하, 우하]. 그림자에 강하게 전역·지역 이진화를 둘 다 쓰고,
    3개만 찾으면(그림자에 묻히거나 사진 끝에 잘리면) 나머지 하나를 추정"""
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    adapt = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, (max(w, h) // 15) | 1, 25)
    cands = []
    for bw in (otsu, adapt):
        for c in cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)[0]:
            x, y, cw, ch = cv2.boundingRect(c)
            if (w * 0.008 < cw < w * 0.06 and w * 0.008 < ch < w * 0.06 and 0.5 < cw / ch < 2.0
                    and cv2.contourArea(c) >= 0.75 * cw * ch and blur[y:y + ch, x:x + cw].mean() <= 110):
                cands.append((x + cw / 2, y + ch / 2))
    found = {}
    for k, (tx, ty) in enumerate(((0, 0), (w, 0), (0, h), (w, h))):
        side = [p for p in cands if (p[0] < w / 2) == (tx == 0) and (p[1] < h / 2) == (ty == 0)]
        if side:
            found[k] = min(side, key=lambda p: np.hypot(p[0] - tx, p[1] - ty))
    if len(found) == 3:
        miss = ({0, 1, 2, 3} - set(found)).pop()
        a, b, c = {0: (1, 2, 3), 1: (0, 3, 2), 2: (0, 3, 1), 3: (1, 2, 0)}[miss]
        found[miss] = tuple(np.add(found[a], found[b]) - np.array(found[c]))
    return np.float32([found[k] for k in range(4)]) if len(found) == 4 else None

def ink_outside(flat, q):
    """정답 칸 바깥 주변에 글씨(잉크)가 있는지. 그림자에 강하게, 칸 테두리 같은 가는 직선은 제외 → 잉크 픽셀 수"""
    gray = cv2.cvtColor(flat, cv2.COLOR_BGR2GRAY)
    ax1, ay1, ax2, ay2 = AROUND_BOXES[q]
    bx1, by1, bx2, by2 = ANSWER_BOXES[q]
    ink = cv2.adaptiveThreshold(gray[PY(ay1):PY(ay2), PX(ax1):PX(ax2)], 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                cv2.THRESH_BINARY_INV, 35, 25)
    m = 0.013   # 사진 정렬 오차를 감안한 칸 테두리 여유
    ink[max(PY(by1 - m) - PY(ay1), 0):PY(by2 + m) - PY(ay1), max(PX(bx1 - m) - PX(ax1), 0):PX(bx2 + m) - PX(ax1)] = 0
    ink = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    n, _, st, _ = cv2.connectedComponentsWithStats(ink)
    return int(sum(st[i, 4] for i in range(1, n)
                   if st[i, 4] >= 25 and not (min(st[i, 2:4]) <= 6 and max(st[i, 2:4]) >= 8 * min(st[i, 2:4]))))

def crop_sheet(sid, path):
    """사진을 반듯하게 펴서 저장하고, 반·번호·이름 칸과 정답 칸을 자르고, 칸 밖 글씨를 검사"""
    img = cv2.imread(path)
    if max(img.shape[:2]) > 2200:   # 큰 휴대폰 사진은 먼저 줄여서 빠르게 (결과 해상도에는 충분)
        s_ = 2200 / max(img.shape[:2])
        img = cv2.resize(img, None, fx=s_, fy=s_, interpolation=cv2.INTER_AREA)
    marks = find_corners(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
    if marks is None:   # 스캔 앱으로 시험지 테두리에 맞춰 잘린 사진이면 크기만 맞춤
        flat = cv2.resize(img, (PAGE_W, PAGE_H))
    else:
        M = cv2.getPerspectiveTransform(marks, CORNERS * np.float32([PAGE_W, PAGE_H]))
        flat = cv2.warpPerspective(img, M, (PAGE_W, PAGE_H), borderValue=(255, 255, 255))
    cv2.imwrite(f'work/flat/{sid}.png', flat)
    x1, y1, x2, y2 = ID_BOX
    cv2.imwrite(f'crops/{sid}_id.png', flat[PY(y1):PY(y2), PX(x1):PX(x2)])
    for q, (x1, y1, x2, y2) in ANSWER_BOXES.items():
        cv2.imwrite(f'crops/{sid}_q{q}.png', flat[PY(y1 + INSET):PY(y2 - INSET), PX(x1 + INSET):PX(x2 - INSET)])
        OUTSIDE[(sid, q)] = ink_outside(flat, q) > OUTSIDE_MIN_INK
        if OUTSIDE[(sid, q)]:   # 칸 밖 글씨가 있으면 주변까지 넓게 잘라 둠
            ax1, ay1, ax2, ay2 = AROUND_BOXES[q]
            cv2.imwrite(f'crops/{sid}_q{q}_wide.png', flat[PY(ay1):PY(ay2), PX(ax1):PX(ax2)])
    return flat, marks is not None

OUTSIDE = {}
if IMAGE_MODE:
    flats = {}
    for sid, path in PAGES:
        flats[sid], ok = crop_sheet(sid, path)
        out_q = [q for q in ANSWER_BOXES if OUTSIDE[(sid, q)]]
        print(('   ' if ok else '⚠️ ') + sid + ('' if ok else '  모서리 표시를 못 찾음 → 칸이 어긋났을 수 있음')
              + (f'  ✏️ 칸 밖 글씨: {out_q}번' if out_q else ''))
    # 미리보기: 처음 몇 장에 잘라낼 위치 표시 (어긋나면 위 좌표 조정)
    show = list(flats.items())[:4]
    fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 6), squeeze=False)
    for ax, (sid, flat) in zip(axes[0], show):
        vis = cv2.cvtColor(flat, cv2.COLOR_BGR2RGB).copy()
        for name, (x1, y1, x2, y2) in {'id': ID_BOX, **ANSWER_BOXES}.items():
            color = (0, 90, 255) if name == 'id' else ((255, 140, 0) if OUTSIDE.get((sid, name)) else (255, 0, 0))
            cv2.rectangle(vis, (PX(x1), PY(y1)), (PX(x2), PY(y2)), color, 5)
        ax.imshow(vis); ax.axis('off'); ax.set_title(sid)
    plt.suptitle('잘라낼 위치 (파랑: 반·번호·이름, 빨강: 정답 칸, 주황: 칸 밖 글씨 감지)'); plt.tight_layout(); plt.show()

### 4-3. AI 읽기 준비
Gemini에게 **정답을 알려주지 않고** 보이는 그대로 읽게 합니다. 키는 왼쪽 **열쇠 아이콘** → 이름 `GEMINI_API_KEY` → **노트북 액세스** 켜기.
- 키가 없어도 **미리 읽어 둔 결과**(`gemini_cache.json`)를 1-3에서 같이 올리면 그 답안지는 AI 읽기 결과로 채점됩니다. 내 키를 남에게 주지 않고 노트북을 공유하는 방법이에요.
- 키도, 미리 읽어 둔 결과도 없는 답안지는 직접 입력 모드로 넘어갑니다.
- 읽은 결과는 `work/gemini_cache.json`에 저장되니, 공유하려면 이 파일을 내려받아 주면 됩니다.

In [ ]:
READ_PROMPT = """이 이미지는 학생이 시험 답안 칸에 손으로 쓴 글씨입니다.
정답을 추측하거나 맞춤법을 고치지 말고, 보이는 글자를 그대로 옮기세요. JSON으로만 답하세요.
{"text": "읽은 글자 그대로 (아무것도 없으면 빈 문자열)", "messy": true 또는 false,
 "messy_reason": "지운 자국, 답이 두 개, 취소선, 덮어쓰기, 알아볼 수 없음 중 해당하는 것 (없으면 빈 문자열)"}"""
WIDE_PROMPT = """이 이미지는 시험 답안지의 한 문항 부분입니다. 네모 칸이 정답란인데, 학생이 칸 밖에도 글씨를 썼습니다.
X표나 줄을 그어 지운 글씨는 빼고, 학생이 최종 답으로 쓴 글씨를 보이는 그대로 옮기세요. 정답을 추측하거나 고치지 마세요.
JSON으로만 답하세요. {"text": "최종 답으로 보이는 글씨 그대로", "messy": true, "messy_reason": "상황 설명"}"""
SHEET_PROMPT = """이 이미지는 한 학생의 시험 답안지에서 칸들을 잘라 위에서부터 차례로 붙인 것입니다. 각 칸 왼쪽 위에 칸 이름이 적혀 있습니다.
각 칸에 학생이 손으로 쓴 글씨를 보이는 그대로 옮기세요. 정답을 추측하거나 맞춤법을 고치지 마세요.
'(칸 밖 글씨)'라고 적힌 칸은 네모 정답란 밖에도 글씨가 있는 경우이니, X표나 줄을 그어 지운 글씨는 빼고 최종 답만 옮기세요.
JSON으로만 답하세요.
{"class": "반 (없으면 빈 문자열)", "number": "번호", "name": "이름",
 "answers": {"1": {"text": "읽은 글자 그대로 (빈칸이면 빈 문자열)", "messy": true 또는 false, "messy_reason": "지운 자국, 답 두 개, 취소선, 덮어쓰기 등"}, "2": {...}}}"""
ID_PROMPT = """이 이미지는 시험지 머리글의 '( )반 ( )번 이름( )' 부분입니다.
괄호 안에 손으로 쓴 반, 번호, 이름을 보이는 그대로 옮기세요. JSON으로만 답하세요.
{"class": "반 (없으면 빈 문자열)", "number": "번호 (없으면 빈 문자열)", "name": "이름 (없으면 빈 문자열)"}"""

class NoReading(Exception):
    """미리 읽어 둔 결과도 없고 Gemini 키도 없어 읽을 수 없음 (→ 직접 입력)"""

CACHE_PATH = 'work/gemini_cache.json'
CACHE = json.load(open(CACHE_PATH, encoding='utf-8')) if os.path.exists(CACHE_PATH) else {}
# 다른 사람이 미리 읽어 둔 결과(gemini_cache.json)를 1-3에서 같이 올렸으면 합쳐서 씀 → 키 없이도 그 답안지는 AI 읽기 결과로 채점
if os.path.exists('inputs/data/gemini_cache.json'):
    CACHE.update(json.load(open('inputs/data/gemini_cache.json', encoding='utf-8')))
    print(f'미리 읽어 둔 결과 {len(CACHE)}개를 불러왔습니다')
TOKEN_LOG = []   # 호출마다 사용한 토큰 수 (비용 계산용)
READ_ERRORS = {}  # 시험지 → 마지막으로 난 읽기 오류 (왜 못 읽었는지 보여 주기 위해)
import threading
from concurrent.futures import ThreadPoolExecutor as _Pool, TimeoutError as _FuturesTimeout
CACHE_LOCK = threading.Lock()   # 여러 칸을 동시에 읽을 때 저장 파일이 꼬이지 않게
_CALLS = _Pool(max_workers=16)  # 호출을 따로 돌려서, 응답이 안 와도 정해진 시간에 반드시 끊음

client = None
if IMAGE_MODE and USE_GEMINI:
    try:
        try:
            from google.colab import userdata
            client_key = userdata.get('GEMINI_API_KEY')
        except ImportError:
            client_key = os.environ['GEMINI_API_KEY']
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=client_key)
        del client_key   # 키를 노트북 변수에 남겨 두지 않음
        print('Gemini 연결 완료:', GEMINI_MODEL)
    except Exception as e:
        print(f'⚠️ Gemini를 쓸 수 없음({type(e).__name__}) → 미리 읽어 둔 결과가 있는 답안지는 그걸 쓰고, 없으면 직접 입력')
        print('   키를 쓰려면: 왼쪽 열쇠 아이콘 → 이름 GEMINI_API_KEY로 저장 → 노트북 액세스 켜기 → 이 셀 다시 실행')

def pick_model():
    """계정에서 쓸 수 있는 Flash 모델들에 실제 답안지(첫 장)를 읽혀 보고, 제대로 읽으면서 가장 빠른 모델을 고름"""
    global GEMINI_MODEL
    sid0 = PAGES[0][0]
    cells0 = {q: (f'crops/{sid0}_q{q}_wide.png' if OUTSIDE[(sid0, q)] else f'crops/{sid0}_q{q}.png') for q in QS}
    png = open(compose_sheet(sid0, cells0), 'rb').read()
    try:
        names = [m.name.split('/')[-1] for m in client.models.list() if 'generateContent' in (m.supported_actions or [])]
    except Exception as e:
        print('모델 목록을 못 가져와 그대로 씁니다:', str(e)[:100]); return
    flash = [n for n in names if 'flash' in n and not any(w in n for w in ('image', 'tts', 'live', 'audio', 'exp'))]
    cands = list(dict.fromkeys([GEMINI_MODEL] + sorted(flash, reverse=True)))[:6]
    results = []
    for name in cands:
        t0 = time.time()
        def call():
            try:
                return client.models.generate_content(model=name, contents=[types.Part.from_bytes(data=png, mime_type='image/png'), SHEET_PROMPT],
                    config=types.GenerateContentConfig(temperature=TEMPERATURE, response_mime_type='application/json',
                                                       thinking_config=types.ThinkingConfig(thinking_level='MINIMAL')))
            except Exception as e:
                if 'thinking' not in str(e).lower():
                    raise
                return client.models.generate_content(model=name, contents=[types.Part.from_bytes(data=png, mime_type='image/png'), SHEET_PROMPT],
                    config=types.GenerateContentConfig(temperature=TEMPERATURE, response_mime_type='application/json'))
        try:
            r = _CALLS.submit(call).result(timeout=60)
            out = json.loads(re.sub(r'^```(json)?|```$', '', (r.text or '').strip()).strip())
            filled = sum(bool(str((out.get('answers') or {}).get(str(q), {}).get('text', '')).strip()) for q in QS)
            sec = time.time() - t0
            print(f'  ✓ {name:<30} {sec:5.1f}초  읽은 칸 {filled}/{len(QS)}  (반 {out.get("class", "")}, 번호 {out.get("number", "")}, {out.get("name", "")})')
            if filled:
                results.append((sec, name))
        except _FuturesTimeout:
            print(f'  ✗ {name:<30} 60초 안에 응답 없음')
        except Exception as e:
            print(f'  ✗ {name:<30} {str(e)[:90]}')
    if results:
        GEMINI_MODEL = min(results)[1]
        print(f'→ 이 모델로 읽습니다: {GEMINI_MODEL} (한 장 {min(results)[0]:.1f}초)')
    else:
        print('⚠️ 제대로 읽는 모델을 찾지 못했습니다. 위 ✗ 옆 메시지(키, 사용량, 모델)를 확인하세요.')

def gemini_config(timeout=None):
    kw = dict(temperature=TEMPERATURE, response_mime_type='application/json')   # 출력 길이를 줄이면 생각 토큰 때문에 답이 잘릴 수 있어 제한 없음
    if timeout:
        kw['http_options'] = types.HttpOptions(timeout=int(timeout * 1000))
    if READ_THINKING_LEVEL not in ('', 'OFF'):
        kw['thinking_config'] = types.ThinkingConfig(thinking_level=READ_THINKING_LEVEL)
    return types.GenerateContentConfig(**kw)

def ask_gemini(path, prompt, k, sid, cell, deadline=None):
    """이미지 하나를 k번째로 읽힌다 (정답은 알려주지 않음). 읽은 결과는 저장해 두고 다시 부르지 않음.
    deadline(시각)까지 못 읽으면 TimeoutError"""
    global READ_THINKING_LEVEL
    key = f'{os.path.basename(path)}#{k}'
    if key not in CACHE and client is None:
        raise NoReading('미리 읽어 둔 결과가 없고 Gemini 키도 없음')
    if key not in CACHE:
        for t in range(5):
            left = (deadline - time.time()) if deadline else CALL_TIMEOUT
            if left < 2:
                raise TimeoutError('시간 안에 읽지 못함')
            try:
                limit = min(CALL_TIMEOUT, left)
                fut = _CALLS.submit(client.models.generate_content, model=GEMINI_MODEL,
                                    contents=[types.Part.from_bytes(data=open(path, 'rb').read(), mime_type='image/png'), prompt],
                                    config=gemini_config(limit))
                try:
                    resp = fut.result(timeout=limit + 1)
                except _FuturesTimeout:
                    READ_ERRORS[sid] = f'{limit:.0f}초 안에 응답 없음 (모델이 느림 → AUTO_PICK_MODEL 또는 다른 모델)'
                    print(f'   {sid}: {limit:.0f}초 안에 응답이 없어 끊음')
                    continue
                out = json.loads(re.sub(r'^```(json)?|```$', '', (resp.text or '').strip()).strip())
                usage = getattr(resp, 'usage_metadata', None)
                out['_tokens'] = int(getattr(usage, 'total_token_count', 0) or 0)
                with CACHE_LOCK:
                    CACHE[key] = out
                    json.dump(CACHE, open(CACHE_PATH, 'w', encoding='utf-8'), ensure_ascii=False)
                break
            except Exception as e:
                if 'thinking' in str(e).lower() and READ_THINKING_LEVEL != 'OFF':
                    READ_THINKING_LEVEL = 'OFF'   # 이 모델은 생각 단계 설정을 지원하지 않음 → 끄고 다시
                    continue
                msg = str(e) if not isinstance(e, json.JSONDecodeError) else 'AI 답이 JSON 형식이 아님 (답이 중간에 잘렸을 수 있음)'
                READ_ERRORS[sid] = msg[:150]
                wait = 5 * 2 ** t
                m = re.search(r'retry in ([\d.]+)s|retryDelay[^\d]*(\d+)', msg)
                if m:
                    wait = float(m.group(1) or m.group(2)) + 1
                if deadline and time.time() + wait > deadline - 2:   # 기다리면 마감을 넘김 → 이 장은 포기
                    raise TimeoutError(f'시간 안에 읽지 못함 ({msg[:60]})')
                if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                    print(f'   분당 사용량 제한 → {wait:.0f}초 기다렸다 다시 (자주 나오면 1-2의 READ_WORKERS를 줄이세요)')
                else:
                    print(f'   다시 시도 {t + 1}/5: {msg[:100]}')
                time.sleep(min(wait, 30))
        else:
            raise RuntimeError('Gemini 호출 실패: 키, 모델 이름, 사용량 제한을 확인하세요')
    TOKEN_LOG.append({'시험지': sid, '칸': cell, '토큰': CACHE[key].get('_tokens', 0)})
    return CACHE[key]

def compose_sheet(sid, cells):
    """반·번호·이름 칸과 문항 칸들을 위에서부터 이어 붙인 이미지 한 장 (각 칸 위에 이름표)"""
    parts = [('반·번호·이름', f'crops/{sid}_id.png')] + \
            [(f'{q}번' + (' (칸 밖 글씨)' if OUTSIDE[(sid, q)] else ''), cells[q]) for q in QS]
    W = 768   # 비전 모델이 한 번에 보는 크기에 맞춰 토큰과 시간을 줄임
    font = ImageFont.truetype(GOTHIC_PATH, 24)
    tiles = []
    for label, p in parts:
        im = Image.open(p).convert('RGB')
        im = im.resize((W, max(1, int(im.height * W / im.width))))
        tile = Image.new('RGB', (W, im.height + 40), 'white')
        ImageDraw.Draw(tile).text((6, 4), label, font=font, fill=(0, 0, 200))
        tile.paste(im, (0, 40))
        tiles.append(tile)
    out = Image.new('RGB', (W, sum(t.height for t in tiles) + 12 * len(tiles)), (200, 200, 200))
    y = 0
    for t in tiles:
        out.paste(t, (0, y)); y += t.height + 12
    path = f'crops/{sid}_sheet.png'
    out.save(path)
    return path

def borderline(q, text, messy):
    """한 번 더 읽어 볼 만한 칸: 지저분함, 빈칸, 허용 답안과 한두 글자 차이, 헷갈리는 글자만 다름.
    허용 답안과 똑같거나, 등록한 오답이거나, 확실히 다른 답이면 한 번 읽은 걸로 충분"""
    t = normalize(text)
    if messy or not t:
        return True
    if t in accepted(q) or t in {normalize(w) for w in REGISTERED_WRONG.get(q, [])}:
        return False
    return any(edit_distance(t, a) <= 2 or (len(a) == len(t) and all(x == y or is_confusable(x, y) for x, y in zip(t, a)))
               for a in accepted(q))

def read_sheet(sid, cells):
    """한 장을 모아서 적응형으로 읽기: 2번 읽어 모든 칸이 같으면 끝, 하나라도 다르면 3번째 → (반·번호·이름, {문항: 읽기 목록}, {문항: 지저분함})"""
    path = compose_sheet(sid, cells)
    deadline = time.time() + SHEET_TIME_LIMIT
    reads = []
    get = lambda r, q: (r.get('answers') or {}).get(str(q), {}) or {}
    if READ_STRATEGY == 'fast':   # 한 번 읽고, 애매한 칸이 있을 때만 한 번 더
        for k in range(2):
            try:
                reads.append(ask_gemini(path, SHEET_PROMPT, k, sid, 'sheet', deadline))
            except TimeoutError:
                if not reads:
                    raise
                break
            if k == 0 and not any(borderline(q, str(get(reads[0], q).get('text', '')), bool(get(reads[0], q).get('messy')))
                                  for q in QS):
                break
        r0 = reads[0]
        per_q = {q: [str(get(r, q).get('text', '')) for r in reads] for q in QS}
        messy = {q: sum(bool(get(r, q).get('messy')) for r in reads) >= (len(reads) + 1) // 2 for q in QS}
        return (str(r0.get('class', '')), str(r0.get('number', '')), str(r0.get('name', ''))), per_q, messy
    for k in range(MAX_READS):
        try:
            reads.append(ask_gemini(path, SHEET_PROMPT, k, sid, 'sheet', deadline))
        except TimeoutError:
            if not reads:
                raise
            break   # 시간이 모자라면 이미 읽은 결과만 씀
        texts = [{q: normalize(str((r.get('answers') or {}).get(str(q), {}).get('text', ''))) for q in QS} for r in reads]
        if k + 1 >= MIN_READS and all(len({t[q] for t in texts}) == 1 for q in QS):
            break
    first = reads[0]
    per_q, messy = {}, {}
    for q in QS:
        items = [(r.get('answers') or {}).get(str(q), {}) for r in reads]
        n = MIN_READS if len({normalize(str(i.get('text', ''))) for i in items[:MIN_READS]}) == 1 else len(items)
        per_q[q] = [str(i.get('text', '')) for i in items[:n]]          # 앞의 두 번이 같은 칸은 두 번만 씀
        messy[q] = sum(bool(i.get('messy')) for i in items[:n]) >= (n + 1) // 2
    return (str(first.get('class', '')), str(first.get('number', '')), str(first.get('name', ''))), per_q, messy

def read_cell(path, prompt, sid, cell, deadline=None):
    """적응형 읽기: MIN_READS번 읽어 결과가 모두 같으면 끝, 다르면 MAX_READS번까지 더 읽음"""
    reads = []
    for k in range(MAX_READS):
        reads.append(ask_gemini(path, prompt, k, sid, cell, deadline))
        if k + 1 >= MIN_READS and len({normalize(r.get('text', '')) for r in reads}) == 1:
            break
    return reads

QS = [q for q in ANSWER_KEY if q in ANSWER_BOXES] if IMAGE_MODE else list(ANSWER_KEY)
if client is not None and IMAGE_MODE and AUTO_PICK_MODEL:
    print('계정에서 되는 모델을 첫 답안지로 시험해 가장 빠르게 제대로 읽는 모델을 고릅니다 (1~2분, 꺼려면 1-2의 AUTO_PICK_MODEL = False)')
    pick_model()

### 4-4. 읽기
**한 장의 칸들을 이미지 한 장으로 모아** 한 번에 읽고(`READ_MODE = 'sheet'`), 기본 전략(`READ_STRATEGY = 'fast'`)은 **한 번 읽어서 애매한 칸이 없으면 끝**, 애매한 칸(허용 답안과 한두 글자 차이, 헷갈리는 글자, 지저분함, 빈칸)이 있을 때만 한 번 더 읽습니다. 그래서 대부분 **장당 1~2번만 호출**하고, 한 장에 30초를 넘기면 기다리지 않고 교사 확인으로 넘깁니다. 읽기 실수를 더 꼼꼼히 잡으려면 `'safe'`(2~3번 읽기)로 바꾸세요.

- "분당 사용량 제한 → N초 기다렸다 다시"가 자주 나오면 무료 사용량의 분당 한도에 걸린 거예요. 1-2의 `READ_WORKERS`를 2~3으로 줄이세요.
- 칸끼리 서로 영향 없이 따로 읽고 싶으면 `READ_MODE = 'cell'` (느리지만 칸별로 독립)
- 출력의 `*`는 지저분한 칸, `!`는 칸 밖 글씨

**직접 입력 모드**: 시험지마다 잘린 칸 그림 아래 입력창에 두 줄 → `1,3,김철수` / `이순신/*신라대장경//평양/3·1 운동`

In [ ]:
QS = [q for q in ANSWER_KEY if (q in ANSWER_BOXES if IMAGE_MODE else f'{q}번' in STUDENT_DF.columns)]
STUDENTS, ANSWERS = {}, []

def parse_cell(cell):
    """학생 답안 표의 칸 표시: * 지저분함, ! 칸 밖, | 여러 번 읽은 결과 → (읽기 목록, 지저분함, 칸밖)"""
    s, messy, outside = str(cell), False, False
    while s[:1] in ('*', '!'):
        messy, outside = messy or s[0] == '*', outside or s[0] == '!'
        s = s[1:]
    return [t.strip() for t in s.split('|')], messy, outside

def show_crops(sid, paths):
    fig, axes = plt.subplots(len(paths), 1, figsize=(8, 1.1 * len(paths)))
    for ax, (title, p) in zip(np.atleast_1d(axes), paths):
        ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax.axis('off'); ax.set_title(title, loc='left', fontsize=9)
    plt.tight_layout(); plt.show()

def read_sheet_ai_jobs(ex, sid, cells):
    """한 장의 반·번호·이름과 모든 칸을 동시에 읽도록 맡김 → (반·번호·이름 작업, {문항: 작업})"""
    deadline = time.time() + SHEET_TIME_LIMIT * 2   # 칸마다 읽기는 여러 장이 한꺼번에 기다리므로 여유를 둠
    id_job = ex.submit(ask_gemini, f'crops/{sid}_id.png', ID_PROMPT, 0, sid, 'id', deadline)   # 반·번호·이름은 한 번만
    jobs = {q: ex.submit(read_cell, cells[q], WIDE_PROMPT if OUTSIDE[(sid, q)] else READ_PROMPT, sid, f'{q}번', deadline)
            for q in QS}
    return id_job, jobs

def collect_sheet_whole(sid, cells, job):
    (c, n, name), per_q, messy = job.result()
    STUDENTS[sid] = {'반': c, '번호': n, '이름': name}
    return [{'시험지': sid, '문항': q, '읽기': per_q[q], '지저분함': messy[q], '칸밖': OUTSIDE[(sid, q)], '이미지': cells[q]}
            for q in QS]

def collect_sheet(sid, cells, id_job, jobs):
    r = id_job.result()
    STUDENTS[sid] = {'반': str(r.get('class', '')), '번호': str(r.get('number', '')), '이름': str(r.get('name', ''))}
    rows = []
    for q in QS:
        reads = jobs[q].result()
        rows.append({'시험지': sid, '문항': q, '읽기': [str(x.get('text', '')) for x in reads],
                     '지저분함': sum(bool(x.get('messy')) for x in reads) >= (len(reads) + 1) // 2,   # 절반 이상이 지저분하다고 하면
                     '칸밖': OUTSIDE[(sid, q)], '이미지': cells[q]})
    return rows

def read_sheet_manual(sid, cells):
    """직접 입력: 시험지 한 장에 두 줄만"""
    show_crops(sid, [(f'{sid}: 반·번호·이름', f'crops/{sid}_id.png')] +
               [(f'{q}번' + (' (칸 밖 글씨)' if OUTSIDE[(sid, q)] else ''), cells[q]) for q in QS])
    c, n, name = (input('반,번호,이름 (예: 1,3,김철수): ').split(',') + ['', '', ''])[:3]
    STUDENTS[sid] = {'반': c.strip(), '번호': n.strip(), '이름': name.strip()}
    while len(typed := input(f'{len(QS)}개 답을 / 로 구분 (빈칸은 비우고, 지저분한 칸은 앞에 *): ').split('/')) != len(QS):
        print(f'  {len(QS)}개를 / 로 구분해 다시 입력하세요')
    return [{'시험지': sid, '문항': q, '읽기': [t.strip().lstrip('*').strip()], '지저분함': t.strip().startswith('*'),
             '칸밖': OUTSIDE[(sid, q)], '이미지': cells[q]} for q, t in zip(QS, typed)]

if IMAGE_MODE:
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    CELLS = {sid: {q: (f'crops/{sid}_q{q}_wide.png' if OUTSIDE[(sid, q)] else f'crops/{sid}_q{q}.png') for q in QS}
             for sid, _ in PAGES}
    manual = []
    ex = ThreadPoolExecutor(max_workers=max(1, READ_WORKERS))
    try:
        if READ_MODE == 'sheet':   # 한 장의 칸들을 한 이미지로 모아 한 번에 (장당 2~3번 호출)
            pending = {sid: ex.submit(read_sheet, sid, CELLS[sid]) for sid, _ in PAGES}
        else:                      # 칸마다 따로 (장당 11~16번 호출)
            pending = {sid: read_sheet_ai_jobs(ex, sid, CELLS[sid]) for sid, _ in PAGES}
        for k, (sid, _) in enumerate(PAGES, 1):
            try:
                rows = (collect_sheet_whole(sid, CELLS[sid], pending[sid]) if READ_MODE == 'sheet'
                        else collect_sheet(sid, CELLS[sid], *pending[sid]))
                ANSWERS += rows
                print(f'  [{k}/{len(PAGES)}] {sid} 읽음 ({time.time() - t0:.0f}초)')
            except NoReading:   # 키도 없고 미리 읽어 둔 결과도 없는 답안지 → 나중에 직접 입력
                manual.append(sid)
            except TimeoutError:  # 정해진 시간 안에 못 읽은 답안지 → 기다리지 않고 교사 확인으로
                STUDENTS[sid] = {'반': '', '번호': '', '이름': ''}
                ANSWERS += [{'시험지': sid, '문항': q, '읽기': [''], '지저분함': False, '칸밖': OUTSIDE[(sid, q)],
                             '이미지': CELLS[sid][q], '메모': f'AI가 {SHEET_TIME_LIMIT}초 안에 읽지 못함 → 교사가 직접 확인'} for q in QS]
                print(f'  [{k}/{len(PAGES)}] {sid} ⏱ {SHEET_TIME_LIMIT}초 안에 못 읽음 → 교사 확인 ({time.time() - t0:.0f}초)')
    finally:
        ex.shutdown(wait=False, cancel_futures=True)   # 응답이 안 오는 호출을 기다리지 않음
    failed = [a['시험지'] for a in ANSWERS if a.get('메모')]
    failed = list(dict.fromkeys(failed))
    if failed:
        print(f'\n⚠️ AI가 읽지 못한 답안지 {len(failed)}장 (이 장의 칸은 모두 교사 확인으로 감):')
        for sid in failed:
            print(f'   {sid}: {READ_ERRORS.get(sid, "시간 초과")}')
        if len(failed) == len(PAGES):
            raise RuntimeError('모든 답안지를 읽지 못했습니다. 위 이유를 보고 4-3을 다시 실행하세요 '
                               '(모델이 느리면 AUTO_PICK_MODEL=True, 사용량 제한이면 잠시 뒤, 키 문제면 보안 비밀 확인)')
    for sid in manual:
        ANSWERS += read_sheet_manual(sid, CELLS[sid])
    print(f'읽기 {time.time() - t0:.0f}초 (방식 {READ_MODE}, 동시에 {READ_WORKERS}개씩)')
else:
    if STUDENT_DF['시험지'].duplicated().any() if '시험지' in STUDENT_DF.columns else False:
        raise ValueError('학생 답안 표에 시험지 이름이 겹칩니다')
    for i, st in STUDENT_DF.iterrows():
        sid = st.get('시험지') or f'sheet{i + 1:02d}'
        STUDENTS[sid] = {'반': st.get('반', ''), '번호': st.get('번호', ''), '이름': st.get('이름', '')}
        for q in QS:
            reads, messy, outside = parse_cell(st[f'{q}번'])
            ANSWERS.append({'시험지': sid, '문항': q, '읽기': reads, '지저분함': messy, '칸밖': outside, '이미지': None})

for sid, s in STUDENTS.items():
    mine = [a for a in ANSWERS if a['시험지'] == sid]
    print(f"[{sid}] {s['반']}반 {s['번호']}번 {s['이름']}  |  " +
          ' · '.join(f"{a['문항']}번 {'/'.join(normalize(t) or '(빈칸)' for t in a['읽기'])}"
                     + ('*' if a['지저분함'] else '') + ('!' if a['칸밖'] else '') for a in mine))
if TOKEN_LOG:
    tok = pd.DataFrame(TOKEN_LOG)
    print(f"\nGemini 호출 {len(tok)}번 · 토큰 {tok['토큰'].sum():,}개 · 답안지 1장당 평균 {tok.groupby('시험지')['토큰'].sum().mean():,.0f}개")

### 4-5. 학생 식별
**반+번호**로 명렬표에서 찾고 **이름으로 검증**합니다. 마지막에 답안지가 매칭되지 않은 학생과 그 학생 것일 수 있는 답안지를 보여 줍니다.

In [ ]:
students_df = pd.DataFrame([{'시험지': sid, **s} for sid, s in STUDENTS.items()])
students_df[['반', '번호', '이름']] = students_df[['반', '번호', '이름']].apply(lambda c: c.astype(str).str.strip())
students_df['번호'] = students_df['번호'].str.extract(r'(\d+)', expand=False).fillna('')   # '3번' → '3'
students_df['반'] = students_df['반'].str.extract(r'(\d+)', expand=False).fillna('')

def identify(df):
    """반+번호로 명렬표에서 찾고, 이름으로 검증한다 → (상태, 사유)"""
    keys = list(zip(df['반'], df['번호'], df['이름']))
    counts = Counter(keys)
    out = []
    for c, n, name in keys:
        if not (c.isdigit() and n.isdigit()):
            out.append(('교사 확인', '반 또는 번호를 읽지 못함'))
        elif ROSTER is None:
            out.append(('자동 매칭', '명렬표 없음: 읽은 반·번호 사용'))
        elif (int(c), int(n)) not in ROSTER:
            out.append(('교사 확인', f'명렬표에 없는 번호 ({c}반 {n}번)'))
        elif ROSTER[(int(c), int(n))] != name:
            out.append(('교사 확인', f'번호와 이름이 다름 (명렬표: {ROSTER[(int(c), int(n))]}, 읽은 이름: {name or "없음"})'))
        elif counts[(c, n, name)] > 1:
            out.append(('교사 확인', f'같은 학생 답안지가 {counts[(c, n, name)]}장'))
        else:
            out.append(('자동 매칭', '반·번호·이름 일치'))
    return out

students_df['식별'], students_df['식별 사유'] = zip(*identify(students_df))
students_df['학생'] = students_df['반'] + '반 ' + students_df['번호'] + '번 ' + students_df['이름']
print(f"자동 매칭 {sum(students_df['식별'] == '자동 매칭')}장 · 학생 확인 필요 {sum(students_df['식별'] == '교사 확인')}장")
display(students_df[students_df['식별'] == '교사 확인'][['시험지', '학생', '식별 사유']])

# 답안지가 매칭되지 않은 학생 (성적표에서 빠진 학생)과 이름이 같아 그 학생 것일 수 있는 답안지
missing = pd.DataFrame(columns=['학생', '이 학생 것일 수 있는 답안지'])
if ROSTER is not None:
    matched = {(int(c), int(n)) for c, n, s in zip(students_df['반'], students_df['번호'], students_df['식별']) if s == '자동 매칭'}
    unmatched = students_df[students_df['식별'] == '교사 확인']
    missing = pd.DataFrame([{'학생': f'{c}반 {n}번 {name}',
                             '이 학생 것일 수 있는 답안지': ', '.join(unmatched[unmatched['이름'] == name]['시험지']) or '없음 (미제출?)'}
                            for (c, n), name in sorted(ROSTER.items()) if (c, n) not in matched],
                           columns=['학생', '이 학생 것일 수 있는 답안지'])
    print(f'답안지가 매칭되지 않은 학생 {len(missing)}명')
    display(missing)

### 4-6. 채점
판정, 교사 확인 목록, 성적표, 진단(2명 이상이 같은 오답)이 나오고 **`results/채점결과.xlsx`** 한 파일(시트 5개)로 저장됩니다. 교사 확인 문항은 확인 전까지 0점입니다.

In [ ]:
rows = []
by_sheet = students_df.set_index('시험지')
for a in ANSWERS:
    st = by_sheet.loc[a['시험지']]
    rows.append({'시험지': a['시험지'], '학생': st['학생'], '문항': a['문항'], '읽은 답': top_read(a['읽기'])[0] or '(빈칸)',
                 '읽기 결과': ' / '.join(normalize(t) or '(빈칸)' for t in a['읽기']), '칸밖': a['칸밖'],
                 '근처': near_miss(a['읽기'], a['문항']), '이미지': a['이미지'], '메모': a.get('메모', ''),
                 **make_features(a['읽기'], a['문항'], a['지저분함'])})
result = pd.DataFrame(rows)
result['P(정답)'] = model.predict(result[FEATURES].values, verbose=0).ravel().round(3)
result['판정'], result['사유'] = zip(*[decide(p, f, o, n) for p, (_, f), o, n in
                                    zip(result['P(정답)'], result[FEATURES].iterrows(), result['칸밖'], result['근처'])])
note = result['메모'] != ''   # 시간 안에 못 읽은 칸은 무조건 교사 확인
result.loc[note, '판정'], result.loc[note, '사유'] = '교사 확인', result.loc[note, '메모']
result['득점'] = np.where(result['판정'] == '정답', result['문항'].map(POINTS), 0)
print(f'판정 기준: T_HIGH {T_HIGH} · T_LOW {T_LOW} · 엇갈린 읽기 {MIXED_READS_RULE} · 한 글자 차이 교사 확인 {NEAR_MISS_TO_REVIEW}')
display(result[['시험지', '학생', '문항', '읽기 결과', 'P(정답)', '판정', '사유']])

# 교사 확인 목록 (답안)
check_list = result[result['판정'] == '교사 확인'][['시험지', '학생', '문항', '읽기 결과', '사유']]
print(f'교사가 확인할 답안 {len(check_list)}개')
display(check_list)

# 성적표: 확정 점수 + 남은 확인 항목
score = result.groupby('시험지', sort=False).agg(점수=('득점', 'sum'),
                                                교사확인문항=('판정', lambda s: (s == '교사 확인').sum())).reset_index()
score = students_df[['시험지', '학생', '식별', '식별 사유']].merge(score, on='시험지')
score['상태'] = np.where((score['식별'] == '자동 매칭') & (score['교사확인문항'] == 0), '확정', '교사 확인 필요')
pend = result[result['판정'] == '교사 확인'].groupby('시험지')['문항'].apply(lambda s: sum(POINTS[q] for q in s))
score['확인 대기 점수'] = score['시험지'].map(pend).fillna(0).astype(int)
score['만점'] = sum(POINTS[q] for q in QS)
display(score)

# 진단 리포트: 여러 학생이 같은 오답을 쓴 문항 (오개념 신호, 또는 글씨 문제인지 교사가 볼 가치가 있음)
wrongish = result[(result['판정'] != '정답') & (result['읽은 답'] != '(빈칸)')
                  & np.array([a not in accepted(q) for a, q in zip(result['읽은 답'], result['문항'])])]
diag = (wrongish.groupby(['문항', '읽은 답']).size().reset_index(name='학생 수')
        .query('`학생 수` >= 2').sort_values(['문항', '학생 수'], ascending=[True, False]))
diag['정답'] = diag['문항'].map(lambda q: ANSWER_KEY[q][0])
print('진단: 2명 이상이 같은 오답을 쓴 경우')
display(diag[['문항', '정답', '읽은 답', '학생 수']])

c = result['판정'].value_counts()
print(f"답안지 {len(score)}장 · 답안 {len(result)}개: 정답 {c.get('정답', 0)} · 오답 {c.get('오답', 0)} · 교사 확인 {c.get('교사 확인', 0)}"
      f" → 교사는 답안 {len(result)}개 중 {c.get('교사 확인', 0)}개, 학생 식별 {sum(score['식별'] != '자동 매칭')}장만 확인")
# 교사 확인 사유를 짧은 유형으로 (그래프와 빨간펜 메모에 씀)
REASON_KINDS = [('읽지 못함', '시간 초과'), ('두 개', '답 두 개'), ('칸 밖', '칸 밖 글씨'), ('지저분', '지저분한 칸'), ('순서', '순서 바뀜'), ('혼동쌍', '혼동쌍'),
                ('엇갈림', '읽기 엇갈림'), ('한 글자', '한 글자 차이'), ('P(정답)', '애매한 확률')]
result['사유 유형'] = [next((k for w, k in REASON_KINDS if w in r), '기타') if j == '교사 확인' else ''
                    for r, j in zip(result['사유'], result['판정'])]

# 엑셀 한 파일로 저장 (시트: 성적표, 답안별 판정, 교사 확인, 미매칭 학생, 진단)
with pd.ExcelWriter('results/채점결과.xlsx') as xw:
    score.to_excel(xw, sheet_name='성적표', index=False)
    result.drop(columns=FEATURES + ['근처', '이미지', '메모']).to_excel(xw, sheet_name='답안별 판정', index=False)
    check_list.to_excel(xw, sheet_name='교사 확인', index=False)
    missing.to_excel(xw, sheet_name='미매칭 학생', index=False)
    diag.to_excel(xw, sheet_name='진단', index=False)
score.to_csv('results/성적표.csv', index=False, encoding='utf-8-sig')
print('저장: results/채점결과.xlsx (시트 5개), results/성적표.csv')

### 4-7. 그래프 (올린 답안지 기준)
| 그래프 | 보는 법 |
|---|---|
| ① 판정 결과 | 교사가 확인할 양 (주황) |
| ② P(정답) 분포 | 점선 사이가 애매한 답안 |
| ③ 문항별 판정 비율 | 어려운 문항, 애매한 답이 많은 문항 |
| ④ 교사 확인 이유 | 어떤 규칙이 답안을 교사에게 보냈는지 |
| ⑤ 학생 × 문항 히트맵 | 학생별로 어느 칸이 애매한지 (? = 교사 확인) |
| ⑥ 학생별 점수 | 초록 = 확정, 주황 = 확인 결과에 따라 더해질 수 있는 점수 |
| ⑦ 진단 | 여러 학생이 같은 오답 = 오개념 신호 |
| ⑧ 비용 | 답안지 1장당 토큰 (Gemini로 읽었을 때) |

In [ ]:
n_sheet, n_ans = result['시험지'].nunique(), len(result)
print(f'답안지 {n_sheet}장, 답안 {n_ans}개 기준 그래프 (올린 답안지에 따라 모양이 바뀜)')

# ① 판정 결과  ② P(정답) 분포  ③ 문항별 판정 비율  ④ 교사 확인 사유
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
ax = axes[0, 0]
sns.countplot(data=result, x='판정', order=JUDGE_ORDER, hue='판정', palette=JUDGE_COLOR, legend=False, ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, labels=[f'{int(b.get_height())}개\n({b.get_height() / n_ans:.0%})' for b in bars])
ax.set(title='① 판정 결과: 교사는 주황색만 확인', xlabel='', ylabel='답안 수')
ax.set_ylim(0, result['판정'].value_counts().max() * 1.25)
ax = axes[0, 1]
sns.histplot(data=result, x='P(정답)', hue='판정', hue_order=JUDGE_ORDER, palette=JUDGE_COLOR,
             bins=np.linspace(0, 1, 21), multiple='stack', ax=ax)
threshold_lines(ax); ax.set(title=f'② P(정답) 분포 (점선 = 기준값 {T_LOW}, {T_HIGH})', ylabel='답안 수')
ax = axes[1, 0]
sns.histplot(data=result, x='문항', hue='판정', hue_order=JUDGE_ORDER, palette=JUDGE_COLOR, multiple='fill',
             discrete=True, shrink=0.8, ax=ax)
ax.set(title='③ 문항별 판정 비율: 어려운 문항, 애매한 답이 많은 문항', ylabel='비율', xticks=QS)
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='판정')
ax = axes[1, 1]
if (result['판정'] == '교사 확인').any():
    rv = result[result['판정'] == '교사 확인']
    sns.countplot(data=rv, y='사유 유형', order=rv['사유 유형'].value_counts().index, color=JUDGE_COLOR['교사 확인'], ax=ax)
    for bars in ax.containers:
        ax.bar_label(bars)
ax.set(title='④ 교사 확인으로 보낸 이유', xlabel='답안 수', ylabel='')
plt.tight_layout(); plt.savefig('results/6_dashboard.png', dpi=150, bbox_inches='tight'); plt.show()

# ⑤ 학생 × 문항 히트맵 (? = 교사 확인)
rows_ = result.assign(행=result['시험지'] + ' ' + result['학생'])
order = rows_.drop_duplicates('시험지')['행']
heat = rows_.pivot(index='행', columns='문항', values='P(정답)').reindex(order)
mark = rows_.assign(표시=[f'{p:.2f}' + ('?' if j == '교사 확인' else '') for p, j in zip(rows_['P(정답)'], rows_['판정'])]) \
            .pivot(index='행', columns='문항', values='표시').reindex(order)
plt.figure(figsize=(8, min(1.5 + 0.42 * len(heat), 30)))
sns.heatmap(heat, annot=mark if len(heat) <= 40 else False, fmt='', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'P(정답)'})
plt.title('⑤ 학생 × 문항 P(정답)  (? = 교사 확인)'); plt.ylabel('')
plt.savefig('results/7_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

# ⑥ 학생별 점수 (확정 점수 + 교사 확인 결과에 따라 더해질 수 있는 점수)  ⑦ 진단: 여러 학생이 같은 오답
sc = score.assign(행=score['시험지'] + ' ' + score['학생'] + np.where(score['식별'] == '자동 매칭', '', ' [식별 확인]'),
                  최대=score['점수'] + score['확인 대기 점수'])
fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.38 * len(sc) + 1.5)),
                         gridspec_kw={'width_ratios': [1.3, 1]})
ax = axes[0]
sns.barplot(data=sc, y='행', x='최대', color=JUDGE_COLOR['교사 확인'], alpha=0.55, label='교사 확인 대기', ax=ax)
sns.barplot(data=sc, y='행', x='점수', color=JUDGE_COLOR['정답'], label='확정 점수', ax=ax)
for k, (p, w) in enumerate(zip(sc['점수'], sc['확인 대기 점수'])):
    ax.text(p + w + 1, k, f'{p}점' + (f' (+최대 {w})' if w else ''), va='center', fontsize=9)
ax.set(xlim=(0, sc['만점'].max() * 1.22), xlabel='점수', ylabel='', title='⑥ 학생별 점수')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
ax = axes[1]
if len(diag):
    d = diag.head(12).assign(라벨=lambda t: t['문항'].astype(str) + '번 ' + t['읽은 답'] + ' (정답 ' + t['정답'] + ')')
    sns.barplot(data=d, y='라벨', x='학생 수', hue='문항', palette='tab10', dodge=False, ax=ax)
    for bars in ax.containers:
        ax.bar_label(bars)
else:
    ax.text(0.5, 0.5, '2명 이상이 같은 오답을 쓴 경우가 없습니다', ha='center', va='center', fontsize=12, color='gray')
    ax.set_xticks([]); ax.set_yticks([])
ax.set(title='⑦ 진단: 2명 이상이 같은 오답 (오개념 신호)', ylabel='', xlabel='학생 수')
plt.tight_layout(); plt.savefig('results/8_scores_and_diagnosis.png', dpi=150, bbox_inches='tight'); plt.show()

# ⑧ 비용: 답안지 1장당 토큰 (Gemini로 읽었을 때)
if TOKEN_LOG:
    tok = pd.DataFrame(TOKEN_LOG)
    per = tok.groupby('시험지', sort=False).agg(토큰=('토큰', 'sum'), 호출=('토큰', 'size')).reset_index()
    fig, ax = plt.subplots(figsize=(10, 3.5))
    sns.barplot(data=per, x='시험지', y='토큰', color='#4c72b0', ax=ax)
    ax.axhline(per['토큰'].mean(), color='k', ls='--', lw=1)
    ax.set_title(f"⑧ 답안지 1장당 토큰 (평균 {per['토큰'].mean():,.0f}, 호출 평균 {per['호출'].mean():.1f}번)")
    plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig('results/9_cost.png', dpi=150); plt.show()
    if PRICE_PER_1M_TOKENS:
        won = per['토큰'].mean() / 1e6 * PRICE_PER_1M_TOKENS
        print(f'답안지 1장당 약 {won:.1f}원 · 200장이면 약 {won * 200:,.0f}원')
    else:
        print('비용(원)을 보려면 1-2의 PRICE_PER_1M_TOKENS에 모델의 100만 토큰당 가격을 넣으세요')

### 4-8. 채점된 시험지 (빨간펜)
선생님이 채점하듯 **문항 번호에** 표시합니다: 정답은 **동그라미**, 오답은 **빗금**, 교사 확인은 번호 왼쪽에 **"(재확인)"**. 점수 칸에 확정 점수를 쓰고, 교사 확인이 남았으면 "+확인 N문항"을 적습니다. 학생 식별이 필요하면 제목 왼쪽에 "학생 확인 요함". 한 장씩 `results/graded/`에, 전체는 **`results/채점된_시험지.pdf`**로 저장됩니다.

> 결과 내려받기: 왼쪽 파일 탭 → `results` 폴더에서 오른쪽 클릭 → 다운로드

In [ ]:
# 채점된 시험지: 빨간펜으로 문항 번호에 표시
#   정답 = 번호에 동그라미, 오답 = 번호에 빗금, 교사 확인 = 번호 왼쪽에 "(재확인)", 점수 칸에 점수
PEN = (214, 30, 36, 235)
NUMBER_MARKS = {1: (0.086, 0.315, 0.100, 0.332), 2: (0.086, 0.435, 0.100, 0.451), 3: (0.086, 0.554, 0.100, 0.570),
                4: (0.086, 0.673, 0.100, 0.690), 5: (0.086, 0.792, 0.100, 0.809)}   # 시험지 PDF에서 잰 문항 번호 위치

def pen(size):
    return ImageFont.truetype(PEN_FONT_PATH, size)

def mark_sheet(sid):
    flat = cv2.imread(f'work/flat/{sid}.png')
    page = Image.fromarray(cv2.cvtColor(flat, cv2.COLOR_BGR2RGB)).convert('RGBA')
    layer = Image.new('RGBA', page.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    rng = random.Random(sid)   # 같은 시험지는 항상 같은 손맛
    for _, r in result[result['시험지'] == sid].iterrows():
        if r['문항'] not in NUMBER_MARKS:
            continue
        x1, y1, x2, y2 = (PX(v) if k % 2 == 0 else PY(v) for k, v in enumerate(NUMBER_MARKS[r['문항']]))
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if r['판정'] == '정답':        # 번호에 동그라미 (두 번 그려 손으로 그린 느낌)
            for _ in range(2):
                rr = 42 + rng.uniform(-3, 3)
                o = [rng.uniform(-4, 4) for _ in range(2)]
                d.ellipse((cx - rr + o[0], cy - rr * 0.92 + o[1], cx + rr + o[0], cy + rr * 0.92 + o[1]), outline=PEN, width=6)
        elif r['판정'] == '오답':      # 번호에 빗금
            d.line((cx - 34, cy + 40, cx + 30, cy - 42), fill=PEN, width=8)
        else:                          # 번호 왼쪽에 (재확인)
            size = 40
            while size > 14 and d.textlength('(재확인)', font=pen(size)) > x1 - 10:
                size -= 2
            d.text((x1 - 6 - d.textlength('(재확인)', font=pen(size)), cy - size * 0.6), '(재확인)', font=pen(size), fill=PEN)
    st = score[score['시험지'] == sid].iloc[0]
    sx1, sy1, sx2, sy2 = PX(SCORE_BOX[0]), PY(SCORE_BOX[1]), PX(SCORE_BOX[2]), PY(SCORE_BOX[3])
    txt = f"{st['점수']}"
    d.text(((sx1 + sx2 - d.textlength(txt, font=pen(72))) / 2, sy1 - 10), txt, font=pen(72), fill=PEN)
    if st['교사확인문항']:
        sub = f"+확인 {st['교사확인문항']}문항"
        d.text(((sx1 + sx2 - d.textlength(sub, font=pen(28))) / 2, sy2 - 30), sub, font=pen(28), fill=PEN)
    if st['식별'] != '자동 매칭':      # 제목 왼쪽 빈 곳
        d.text((PX(0.095), PY(0.110)), '학생 확인 요함', font=pen(36), fill=PEN)
    out = Image.alpha_composite(page, layer).convert('RGB')
    out.save(f'results/graded/{sid}.png')
    return out

if IMAGE_MODE:
    marked = [mark_sheet(sid) for sid, _ in PAGES]
    marked[0].save('results/채점된_시험지.pdf', save_all=True, append_images=marked[1:])
    print(f'채점된 시험지 {len(marked)}장 → results/graded/, 한 파일로 results/채점된_시험지.pdf')
    for k in range(0, len(marked), 3):
        part = marked[k:k + 3]
        fig, axes = plt.subplots(1, len(part), figsize=(7 * len(part), 10), squeeze=False)
        for ax, (sid, _), im in zip(axes[0], PAGES[k:k + 3], part):
            ax.imshow(im); ax.axis('off'); ax.set_title(sid, fontsize=14)
        plt.tight_layout(); plt.show()
else:
    print('학생 답안 표로 채점해서 그릴 시험지 이미지가 없습니다 (답안지 사진을 올리면 채점된 시험지가 나옵니다)')

### 4-9. 보고서용 요약
정리 문서 7절의 발표 문장 틀에 맞춰 숫자를 채워 줍니다. `results/보고서_요약.txt`로도 저장됩니다.

In [ ]:
# 보고서에 그대로 쓸 수 있는 요약 (정리 문서 7절의 발표 문장 틀)
c = result['판정'].value_counts()
n_auto = c.get('정답', 0) + c.get('오답', 0)
lines = [
    f"[올린 답안지 기준] 답안지 {result['시험지'].nunique()}장, 답안 {len(result)}개 중 {n_auto / len(result):.0%}({n_auto}개)는 자동 채점했고, "
    f"교사는 {c.get('교사 확인', 0)}개만 확인하면 된다. 학생 식별 확인 {sum(score['식별'] != '자동 매칭')}장.",
    f"[교사 확인 이유] " + ', '.join(f'{k} {v}개' for k, v in result.loc[result['판정'] == '교사 확인', '사유 유형'].value_counts().items()),
]
if 'seed_df' in globals():
    lines.append(f"[시뮬레이션, 시드 {len(seed_df)}개 범위] 정확도 {seed_df['정확도(%)'].min():.1f}~{seed_df['정확도(%)'].max():.1f}%, "
                 f"자동 처리 {seed_df['자동 처리 비율(%)'].min():.0f}~{seed_df['자동 처리 비율(%)'].max():.0f}%, "
                 f"자동 처리 중 오류 {seed_df['자동 처리 중 오류(건)'].min():.0f}~{seed_df['자동 처리 중 오류(건)'].max():.0f}건, "
                 f"ECE {seed_df['ECE'].min():.3f}~{seed_df['ECE'].max():.3f}")
lines.append(f"[보정] 신경망 ECE {ece:.3f} (단순 규칙 {ece_rule:.3f})")
if TOKEN_LOG:
    tok = pd.DataFrame(TOKEN_LOG)
    per = tok.groupby('시험지')['토큰'].sum().mean()
    lines.append(f"[비용] 답안지 1장당 Gemini 호출 평균 {tok.groupby('시험지').size().mean():.1f}번, 토큰 {per:,.0f}개"
                 + (f", 약 {per / 1e6 * PRICE_PER_1M_TOKENS:.1f}원" if PRICE_PER_1M_TOKENS else ''))
lines.append('※ 시뮬레이션 수치는 실제 손글씨 성능이 아니며, 올린 답안지 수가 적으면 일반화에 한계가 있음')
text = '\n'.join(lines)
print(text)
open('results/보고서_요약.txt', 'w', encoding='utf-8').write(text)

---
## 5단계. 시연 웹페이지와 연결 (이 노트북을 API로 열기)
지금까지의 채점 과정(4-2 사진 펴기 → 4-4 AI 읽기 → 4-6 판정 → 4-8 빨간펜)을 **이 노트북 안에서 그대로 웹 API로 엽니다.** 시연 웹페이지는 겉모습만 보여 주고, 채점은 이 노트북의 코드와 신경망이 합니다.

- **4-1에서 올린 답안지가 웹페이지의 "예시 답안지"**로 나오고, 4-4에서 이미 읽어 둔 결과를 쓰므로 **누르면 바로** 채점됩니다.
- "내 답안지 찍어 올리기"로 새 사진을 올리면 4-4와 같은 방식(한 장 최대 30초)으로 읽습니다.
- API 키는 이 노트북(보안 비밀)에만 있고, 웹페이지를 보는 사람에게는 넘어가지 않습니다.
- 이 노트북 런타임이 켜져 있는 동안만 동작합니다.

### 5-1. 웹페이지 올리기 (선택)
`시연페이지_index.html`을 올리면 아래 주소로 바로 웹페이지가 열립니다. 웹페이지를 Vercel 등에 따로 올렸다면 건너뛰고, 그 페이지 주소 뒤에 `?api=시연주소`를 붙이세요.

In [ ]:
# 웹페이지(시연페이지_index.html)와 다른 과목 자료(과목_자료.zip: 과목별 정답표와 예시 답안지)를 올립니다. 둘 다 선택
for p in upload_to('inputs/web', '시연페이지_index.html, 과목_자료.zip을 고르세요 (따로 올렸거나 필요 없으면 창을 닫아도 됨)'):
    if p.lower().endswith('.html') and os.path.basename(p) != 'index.html':
        shutil.copy(p, 'inputs/web/index.html')
    elif p.lower().endswith('.zip'):
        shutil.rmtree('inputs/subjects', ignore_errors=True)   # 다시 올리면 새 자료로 바꿈
        extract_zip(p, 'inputs')   # → inputs/subjects/영어/정답표.csv, inputs/subjects/영어/sheets/...
if os.path.isdir('inputs/subjects'):
    print('과목 자료:', sorted(d for d in os.listdir('inputs/subjects') if os.path.isdir(f'inputs/subjects/{d}')))

### 5-2. 노트북 API 만들기
웹페이지가 부르는 주소(`/v1/subjects`, `/v1/samples`, `/v1/grade/images`, `/v1/jobs/…`, 선생님 판정 `/v1/runs/…/decisions`)를 노트북 함수로 만듭니다.
- 다른 과목을 올렸으면 과목마다 신경망을 학습하고(과목당 10~20초), **예시 답안지를 미리 읽어 둬서** 웹페이지에서 누르면 바로 채점됩니다.
- 시연용 접근 코드나 하루 사용 상한은 맨 위에서 바꾸세요.

In [ ]:
# 노트북의 채점 과정(4-2 ~ 4-8과 같은 함수)을 그대로 웹 API로 엽니다. 시연 웹페이지는 이 API를 부릅니다.
import threading, uuid, datetime
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, HTMLResponse

# 이 노트북에서 1~4단계로 채점한 과목 이름 (웹페이지 과목 단추에 나옴): 정답표 파일 이름에서 알아내고, 없으면 '역사'
SUBJECT_NAME = next((n for n in ('국어', '영어', '수학', '과학', '역사') if n in TABLE_NAMES.get('정답표', '')), '역사')
print('이 노트북의 과목:', SUBJECT_NAME)
DEMO_ACCESS_CODE = ''   # 정하면 웹페이지의 '내 답안지 올리기'에 이 코드가 필요 (예: 'teamj')
DEMO_DAILY_LIMIT = 40   # 하루에 새로 읽는 답안지 수 상한 (키 사용량 보호)
PREFETCH_SUBJECTS = True   # 다른 과목 예시 답안지를 지금 미리 읽어 둠 → 웹페이지에서 누르면 바로 채점됨
DEMO_READ_STRATEGY = 'fast'   # 웹페이지에서 새로 올린 사진만 빠르게 (예시 답안지는 1-2의 READ_STRATEGY로 미리 읽음)
DEMO_SHEET_TIME_LIMIT = 30    # 웹페이지에서 새로 올린 사진은 한 장 최대 30초
API_RUNS = 'work/api_runs'
os.makedirs(API_RUNS, exist_ok=True)
API_LOCK = threading.Lock()   # 노트북 전역 변수를 함께 쓰므로 한 번에 한 채점씩
JOBS, USAGE = {}, {'date': '', 'sheets': 0}
EN = {'정답': 'correct', '오답': 'wrong', '교사 확인': 'review'}
KO = {v: k for k, v in EN.items()}
RUNS = {}   # 채점 결과 (선생님이 확인 문항을 정답/오답으로 정하면 여기서 다시 계산)
from contextlib import contextmanager

def parse_key(df):
    """정답표 표 → (인정 답, 등록오답, 배점). 1-4와 같은 규칙으로 검사"""
    df = _clean(df)
    miss = [c for c in ('문항', '배점', '정답') if c not in df.columns]
    if miss:
        raise ValueError(f'정답표에 {miss} 열이 없습니다')
    for c in ('허용답안', '등록오답'):
        if c not in df.columns:
            df[c] = ''
    bad = [r['문항'] for _, r in df.iterrows() if not (r['문항'].isdigit() and r['배점'].isdigit() and r['정답'])]
    if bad:
        raise ValueError(f'정답표 문항 {bad}의 번호·배점·정답을 확인하세요')
    key = {int(r['문항']): [r['정답']] + _split(r['허용답안']) for _, r in df.iterrows()}
    return key, {int(r['문항']): _split(r['등록오답']) for _, r in df.iterrows()}, {int(r['문항']): int(r['배점']) for _, r in df.iterrows()}

SUBJECTS = {SUBJECT_NAME: {'key': ANSWER_KEY, 'wrong': REGISTERED_WRONG, 'points': POINTS, 'model': model, 'pages': list(PAGES),
                           'val_accuracy': float(h['val_accuracy'][best]), 'epochs': len(h)}}

@contextmanager
def use_subject(name):
    """노트북 함수들이 쓰는 전역 정답표·신경망을 잠깐 그 과목 것으로 바꿈"""
    global ANSWER_KEY, REGISTERED_WRONG, POINTS, QS, model
    old = (ANSWER_KEY, REGISTERED_WRONG, POINTS, QS, model)
    sub = SUBJECTS[name]
    ANSWER_KEY, REGISTERED_WRONG, POINTS, model = sub['key'], sub['wrong'], sub['points'], sub['model']
    QS = [q for q in ANSWER_KEY if q in ANSWER_BOXES]
    try:
        yield sub
    finally:
        ANSWER_KEY, REGISTERED_WRONG, POINTS, QS, model = old

# 5-1에서 올린 과목 자료(inputs/subjects/과목/정답표.csv + sheets/): 과목마다 그 정답표로 학습 데이터를 만들어 신경망을 따로 학습
# 이 노트북에서 채점한 과목(SUBJECT_NAME)과 같은 과목이면 학습은 건너뛰고 예시 답안지만 더함
used = {sid for sid, _ in PAGES}
SAMPLE_ORIGIN = {sid: (SUBJECT_NAME, sid) for sid, _ in PAGES}   # 시험지 → (과목, 원래 파일 이름): 베이스라인 표에서 정답과 맞춰 봄

def add_samples(name, sheet_dir):
    if not os.path.isdir(sheet_dir):
        return
    os.makedirs(f'work/pages/{name}', exist_ok=True)
    files = sorted(f for f in os.listdir(sheet_dir)
                   if not (name == SUBJECT_NAME and os.path.splitext(f)[0] in used))   # 4-1에서 이미 올린 답안지는 건너뜀
    for stem, path in expand([f'{sheet_dir}/{f}' for f in files], f'work/pages/{name}'):
        sid = f'{name}_{stem}'   # 과목이 달라도 학생 이름이 같을 수 있어 과목 이름을 붙임
        used.add(sid)
        SAMPLE_ORIGIN[sid] = (name, stem)
        SUBJECTS[name]['pages'].append((sid, path))

def prefetch(name, pages):
    """예시 답안지를 미리 읽어 둠 → 웹페이지에서 누르면 바로 채점됨"""
    if not (PREFETCH_SUBJECTS and pages and client is not None):
        return
    t0 = time.time()
    with use_subject(name):
        for sid, path in pages:
            crop_sheet(sid, path)
        pre = ThreadPoolExecutor(max_workers=max(1, READ_WORKERS))
        futs = [pre.submit(read_sheet, sid, {q: (f'crops/{sid}_q{q}_wide.png' if OUTSIDE[(sid, q)] else f'crops/{sid}_q{q}.png')
                                              for q in QS}) for sid, _ in pages]
        ok = 0
        for f_ in futs:
            try:
                f_.result(); ok += 1
            except (TimeoutError, NoReading, RuntimeError):
                pass
        pre.shutdown(wait=False, cancel_futures=True)
    print(f'   예시 답안지 {ok}/{len(pages)}장 미리 읽음 ({time.time() - t0:.0f}초)')

for name in sorted(os.listdir('inputs/subjects')) if os.path.isdir('inputs/subjects') else []:
    folder = f'inputs/subjects/{name}'
    if not os.path.isdir(folder):
        continue
    if name == SUBJECT_NAME:   # 노트북에서 이미 학습한 과목: 예시 답안지만 더함
        before = len(SUBJECTS[name]['pages'])
        add_samples(name, f'{folder}/sheets')
        print(f'{name}: 노트북 과목에 예시 답안지 {len(SUBJECTS[name]["pages"]) - before}장 추가')
        prefetch(name, SUBJECTS[name]['pages'][before:])
        continue
    keys = [f for f in os.listdir(folder) if f.lower().endswith(TABLE_EXT)]
    if not keys:
        continue
    try:
        key, wrong, points = parse_key(read_tables(f'{folder}/{keys[0]}')[0][1])
    except Exception as e:
        print(f'⚠️ {name}: 정답표를 읽지 못해 건너뜀 ({e})')
        continue
    SUBJECTS[name] = {'key': key, 'wrong': wrong, 'points': points, 'model': None, 'pages': []}
    t0 = time.time()
    with use_subject(name) as sub:
        d_ = make_dataset(N_SAMPLES, SEED)
        m_ = build_model()
        hist = pd.DataFrame(train(m_, d_[FEATURES].values, d_['정답여부'].values).history)
        sub.update(model=m_, val_accuracy=float(hist['val_accuracy'].max()), epochs=len(hist))
    add_samples(name, f'{folder}/sheets')
    print(f'{name}: 정답표 {len(key)}문항, 예시 답안지 {len(SUBJECTS[name]["pages"])}장, 신경망 학습 {time.time() - t0:.0f}초 '
          f'(검증 정확도 {SUBJECTS[name]["val_accuracy"]:.1%})')
    prefetch(name, SUBJECTS[name]['pages'])

def api_grade(pairs, subject, progress=None):
    """pairs: [(시험지 이름, 페이지 이미지 경로)] → 웹페이지가 쓰는 결과. 노트북의 crop_sheet, read_sheet, identify,
    make_features, model, decide, mark_sheet를 그대로 쓴다 (예시 답안지는 4-4에서 읽어 둔 결과를 재사용해 바로 끝남)"""
    run_id = uuid.uuid4().hex[:12]
    run = f'{API_RUNS}/{run_id}'
    os.makedirs(run)
    aligned = {sid: crop_sheet(sid, path)[1] for sid, path in pairs}
    cells = {sid: {q: (f'crops/{sid}_q{q}_wide.png' if OUTSIDE[(sid, q)] else f'crops/{sid}_q{q}.png') for q in QS}
             for sid, _ in pairs}
    students, answers, log_start, cache_before = {}, [], len(TOKEN_LOG), set(CACHE)
    if progress:
        progress(0, len(pairs), 'AI가 읽는 중')
    ex = ThreadPoolExecutor(max_workers=max(1, READ_WORKERS))
    try:
        jobs = {sid: ex.submit(read_sheet, sid, cells[sid]) for sid, _ in pairs}
        for k, (sid, _) in enumerate(pairs, 1):
            try:
                (c, n, name), per_q, messy = jobs[sid].result()
                students[sid] = {'반': c, '번호': n, '이름': name}
                answers += [{'시험지': sid, '문항': q, '읽기': per_q[q], '지저분함': messy[q], '칸밖': OUTSIDE[(sid, q)],
                             '이미지': cells[sid][q], '메모': ''} for q in QS]
            except (TimeoutError, NoReading) as e:
                note = (f'AI가 {SHEET_TIME_LIMIT}초 안에 읽지 못함 → 교사가 직접 확인' if isinstance(e, TimeoutError)
                        else 'AI 읽기 키가 없어 읽지 못함 → 교사가 직접 확인')
                students[sid] = {'반': '', '번호': '', '이름': ''}
                answers += [{'시험지': sid, '문항': q, '읽기': [''], '지저분함': False, '칸밖': OUTSIDE[(sid, q)],
                             '이미지': cells[sid][q], '메모': note} for q in QS]
            if progress:
                progress(k, len(pairs), f'{sid} 읽음')
    finally:
        ex.shutdown(wait=False, cancel_futures=True)

    # 학생 식별 (4-5와 같음)
    sdf = pd.DataFrame([{'시험지': sid, **s} for sid, s in students.items()])
    for col in ('반', '번호'):
        sdf[col] = sdf[col].astype(str).str.extract(r'(\d+)', expand=False).fillna('')
    sdf['이름'] = sdf['이름'].astype(str).str.strip()
    sdf['식별'], sdf['식별 사유'] = zip(*identify(sdf))
    sdf['학생'] = sdf['반'] + '반 ' + sdf['번호'] + '번 ' + sdf['이름']
    by = sdf.set_index('시험지')
    # 채점 (4-6과 같음)
    rows = [{'시험지': a['시험지'], '학생': by.loc[a['시험지'], '학생'], '문항': a['문항'], '읽은 답': top_read(a['읽기'])[0] or '(빈칸)',
             '읽기': a['읽기'], '칸밖': a['칸밖'], '지저분': a['지저분함'], '근처': near_miss(a['읽기'], a['문항']),
             '이미지': a['이미지'], '메모': a['메모'], **make_features(a['읽기'], a['문항'], a['지저분함'])} for a in answers]
    res = pd.DataFrame(rows)
    res['P(정답)'] = model.predict(res[FEATURES].values, verbose=0).ravel().round(3)
    res['판정'], res['사유'] = zip(*[decide(p, f, o, nm) for p, (_, f), o, nm in
                                  zip(res['P(정답)'], res[FEATURES].iterrows(), res['칸밖'], res['근처'])])
    note = res['메모'] != ''
    res.loc[note, '판정'], res.loc[note, '사유'] = '교사 확인', res.loc[note, '메모']
    res['득점'] = np.where(res['판정'] == '정답', res['문항'].map(POINTS), 0)
    res['선생님'] = ''   # 선생님이 확인 문항을 정답/오답으로 정하면 여기에 적음
    RUNS[run_id] = {'subject': subject, 'pairs': pairs, 'res': res, 'sdf': sdf, 'aligned': aligned, 'version': 0,
                    'reading': {'reads': len(TOKEN_LOG) - log_start, 'new_calls': len(set(CACHE) - cache_before),
                                'cached': len(TOKEN_LOG) - log_start - len(set(CACHE) - cache_before),
                                'tokens': int(sum(CACHE[k].get('_tokens', 0) for k in set(CACHE) - cache_before)),
                                'model': GEMINI_MODEL if client else None}}
    return render_run(run_id, [sid for sid, _ in pairs])

def render_run(run_id, remark):
    """저장된 채점 결과 → 성적표 계산, remark에 든 시험지의 빨간펜 그림 다시 그리기 → 웹페이지가 쓰는 결과"""
    global result, score
    R = RUNS[run_id]
    run, res, sdf = f'{API_RUNS}/{run_id}', R['res'], R['sdf']
    eff = res['판정'].where(res['선생님'] == '', res['선생님'])   # 선생님이 정한 것이 우선
    res = res.assign(판정=eff)
    res['득점'] = np.where(res['판정'] == '정답', res['문항'].map(POINTS), 0)
    sc = res.groupby('시험지', sort=False).agg(점수=('득점', 'sum'), 교사확인문항=('판정', lambda s: (s == '교사 확인').sum())).reset_index()
    sc = sdf[['시험지', '학생', '식별', '식별 사유']].merge(sc, on='시험지')
    pend = res[res['판정'] == '교사 확인'].groupby('시험지')['문항'].apply(lambda s: sum(POINTS[q] for q in s))
    sc['확인 대기 점수'] = sc['시험지'].map(pend).fillna(0).astype(int)
    if remark:   # 빨간펜 채점 시험지 (4-8의 mark_sheet가 전역 result, score를 쓰므로 잠깐 바꿨다가 되돌림)
        R['version'] += 1
        saved = (globals().get('result'), globals().get('score'))
        result, score = res, sc
        try:
            for sid in remark:
                mark_sheet(sid).save(f"{run}/graded_{sid}_v{R['version']}.png")
                R.setdefault('img', {})[sid] = f"/files/{run}/graded_{sid}_v{R['version']}.png"
        finally:
            result, score = saved
        imgs = [Image.open(f"{run}/graded_{sid}_v{R['img'][sid].rsplit('_v', 1)[1]}") for sid, _ in R['pairs']]
        imgs[0].save(f'{run}/graded.pdf', save_all=True, append_images=imgs[1:])
    review = []
    for _, r in res[(R['res']['판정'] == '교사 확인').values].iterrows():
        crop = f"{run}/crop_{r['시험지']}_q{r['문항']}.png"
        if not os.path.exists(crop):
            shutil.copy(r['이미지'], crop)
        review.append({'sheet_id': r['시험지'], 'student': r['학생'], 'question': int(r['문항']), 'reads': r['읽기'],
                       'reason': R['res'].loc[_, '사유'], 'image_url': f'/files/{crop}',
                       'teacher_decision': EN.get(r['선생님']) if r['선생님'] else None})
    max_score = sum(POINTS[q] for q in QS)
    matched = {(int(c), int(n)) for c, n, s in zip(sdf['반'], sdf['번호'], sdf['식별']) if s == '자동 매칭'}
    unmatched = sdf[sdf['식별'] != '자동 매칭']
    missing_list = [] if ROSTER is None else [
        {'class_no': c, 'number': n, 'name': nm, 'candidate_sheets': unmatched[unmatched['이름'] == nm]['시험지'].tolist()}
        for (c, n), nm in sorted(ROSTER.items()) if (c, n) not in matched]
    cnt = R['res']['판정'].value_counts()
    sub = SUBJECTS[R['subject']]
    return {
        'run_id': run_id, 'subject': R['subject'], 'can_decide': True,
        'summary': {'sheets': len(R['pairs']), 'answers': len(res), 'correct': int(cnt.get('정답', 0)), 'wrong': int(cnt.get('오답', 0)),
                    'review': int(cnt.get('교사 확인', 0)), 'auto_rate': float((cnt.get('정답', 0) + cnt.get('오답', 0)) / max(len(res), 1)),
                    'identification_review': int((sdf['식별'] != '자동 매칭').sum()),
                    'confirmed_sheets': int(((sc['식별'] == '자동 매칭') & (sc['교사확인문항'] == 0)).sum()),
                    'teacher_decided': int((R['res']['선생님'] != '').sum())},
        'answers': [{'sheet_id': r['시험지'], 'question': int(r['문항']), 'read_answer': r['읽은 답'], 'reads': r['읽기'],
                     'messy': bool(r['지저분']), 'outside': bool(r['칸밖']), 'p_correct': float(r['P(정답)']),
                     'decision': EN[R['res'].loc[_, '판정']], 'final': EN[r['판정']], 'reason': R['res'].loc[_, '사유']}
                    for _, r in res.iterrows()],
        'review_items': review,
        'scores': [{'sheet_id': r['시험지'], 'student': r['학생'], 'identification': 'matched' if r['식별'] == '자동 매칭' else 'review',
                    'identification_reason': r['식별 사유'], 'score': int(r['점수']), 'max_score': max_score,
                    'review_count': int(r['교사확인문항']), 'pending_points': int(r['확인 대기 점수']),
                    'status': 'confirmed' if r['식별'] == '자동 매칭' and r['교사확인문항'] == 0 else 'needs_review'} for _, r in sc.iterrows()],
        'missing_students': missing_list,
        'sheet_images': [{'sheet_id': sid, 'aligned': bool(R['aligned'][sid]), 'outside_questions': [q for q in QS if OUTSIDE[(sid, q)]],
                          'graded_url': R['img'][sid]} for sid, _ in R['pairs']],
        'pdf_url': f'/files/{run}/graded.pdf?v={R["version"]}',
        'reading': R['reading'],
        'model_info': {'source': 'notebook', 'val_accuracy': sub.get('val_accuracy'), 'epochs': sub.get('epochs')},
        'warnings': [],
    }

app = FastAPI(title='손글씨 채점 AI (노트북 API)')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['GET', 'POST'], allow_headers=['*'])

def _usage():
    today = datetime.date.today().isoformat()
    if USAGE['date'] != today:
        USAGE.update(date=today, sheets=0)
    return USAGE

@app.get('/', response_class=HTMLResponse)
def page():
    path = 'inputs/web/index.html'
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    return '<p>시연 웹페이지(시연페이지_index.html)를 5-1에서 올리거나, 따로 올린 페이지 주소 뒤에 ?api=이 주소 를 붙이세요.</p>'

@app.get('/v1/config')
def config():
    return {'live_reading': client is not None, 'vision_model': GEMINI_MODEL if client else None,
            'daily_limit': DEMO_DAILY_LIMIT, 'used_today': _usage()['sheets'], 'access_code_required': bool(DEMO_ACCESS_CODE)}

@app.get('/v1/subjects')
def subjects():
    return [{'name': n, 'questions': len(s['key']), 'samples': len(s['pages'])} for n, s in SUBJECTS.items()]

@app.get('/v1/samples')
def samples(subject: str = ''):
    """과목별 예시 답안지 (기본 과목은 4-1에서 올린 답안지: 4-4에서 이미 읽어 둬서 바로 채점됨)"""
    name = {r['시험지']: r['학생'] for _, r in students_df.iterrows()} if 'students_df' in globals() else {}
    return [{'id': sid, 'subject': n, 'title': name.get(sid) or SAMPLE_ORIGIN.get(sid, (n, sid))[1],
             'image_url': f'/files/{path}'}
            for n, s in SUBJECTS.items() if not subject or n == subject for sid, path in s['pages']]

@app.get('/files/{path:path}')
def files_(path: str):
    full = os.path.normpath(path)
    if not full.startswith(('work/pages/', 'work/api_runs/')) or not os.path.isfile(full):
        raise HTTPException(404)
    return FileResponse(full)

def _start(pairs, subject, uploaded, t_high, t_low, mixed, near):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {'status': 'running', 'progress': {'done': 0, 'total': len(pairs), 'message': '준비 중'}}
    def work():
        global T_HIGH, T_LOW, MIXED_READS_RULE, NEAR_MISS_TO_REVIEW, READ_STRATEGY, SHEET_TIME_LIMIT
        with API_LOCK:
            old = (T_HIGH, T_LOW, MIXED_READS_RULE, NEAR_MISS_TO_REVIEW, READ_STRATEGY, SHEET_TIME_LIMIT)
            if uploaded:
                READ_STRATEGY, SHEET_TIME_LIMIT = DEMO_READ_STRATEGY, DEMO_SHEET_TIME_LIMIT
            T_HIGH, T_LOW = t_high, t_low
            MIXED_READS_RULE, NEAR_MISS_TO_REVIEW = ('strict' if mixed else 'off'), near
            try:
                with use_subject(subject):
                    JOBS[job_id].update(status='done', result=api_grade(
                        pairs, subject, lambda d, t, m: JOBS[job_id].update(progress={'done': d, 'total': t, 'message': m})))
            except Exception as e:
                if uploaded:
                    _usage()['sheets'] -= len(pairs)
                JOBS[job_id].update(status='error', error=f'{type(e).__name__}: {e}')
            finally:
                T_HIGH, T_LOW, MIXED_READS_RULE, NEAR_MISS_TO_REVIEW, READ_STRATEGY, SHEET_TIME_LIMIT = old
    threading.Thread(target=work, daemon=True).start()
    return job_id

@app.post('/v1/grade/images')
async def grade_images(sheets: list[UploadFile] = File([]), tables: list[UploadFile] = File([]), sample_ids: str = Form(''),
                       access_code: str = Form(''), t_high: float = Form(0.8), t_low: float = Form(0.2),
                       mixed_reads_to_review: bool = Form(True), near_miss_to_review: bool = Form(True),
                       background: bool = Form(True), subject: str = Form('')):
    if t_low >= t_high:
        raise HTTPException(422, 't_low는 t_high보다 작아야 합니다')
    subject = subject or SUBJECT_NAME
    if subject not in SUBJECTS:
        raise HTTPException(404, f'없는 과목입니다: {subject}')
    known = {sid: (n, path) for n, s in SUBJECTS.items() for sid, path in s['pages']}
    if sample_ids.strip():
        ids = [i.strip() for i in sample_ids.split(',') if i.strip()]
        if any(i not in known for i in ids):
            raise HTTPException(404, '없는 예시 답안지가 있습니다')
        subs = {known[i][0] for i in ids}
        if len(subs) > 1:
            raise HTTPException(422, '한 번에 한 과목의 답안지만 채점할 수 있습니다')
        subject = subs.pop()
        pairs, uploaded = [(i, known[i][1]) for i in ids], False
    else:
        if DEMO_ACCESS_CODE and access_code != DEMO_ACCESS_CODE:
            raise HTTPException(403, '시연용 접근 코드가 맞지 않습니다')
        tag = uuid.uuid4().hex[:6]
        os.makedirs(f'work/api_uploads/{tag}', exist_ok=True)
        os.makedirs(f'work/pages/up{tag}', exist_ok=True)
        paths = []
        for f in sheets:
            p = f'work/api_uploads/{tag}/{os.path.basename(f.filename or "sheet.jpg")}'
            open(p, 'wb').write(await f.read())
            paths.append(p)
        pairs = [(f'up{tag}_{sid}', path) for sid, path in expand(paths, f'work/pages/up{tag}')]   # 예시 답안지와 이름이 겹쳐도 섞이지 않게
        if not pairs:
            raise HTTPException(422, '답안지 사진(jpg, png, heic), PDF, ZIP 중 하나를 올려 주세요')
        if _usage()['sheets'] + len(pairs) > DEMO_DAILY_LIMIT:
            raise HTTPException(429, f'오늘 새로 읽을 수 있는 답안지 수({DEMO_DAILY_LIMIT}장)를 넘었습니다. 예시 답안지로 해 보세요')
        _usage()['sheets'] += len(pairs)
        uploaded = True
    job_id = _start(pairs, subject, uploaded, t_high, t_low, mixed_reads_to_review, near_miss_to_review)
    return {'job_id': job_id, 'status': 'running'}

from pydantic import BaseModel

class Decision(BaseModel):
    sheet_id: str
    question: int
    decision: str   # 'correct' | 'wrong' | 'undo'

@app.post('/v1/runs/{run_id}/decisions')
def decide_by_teacher(run_id: str, d: Decision):
    """선생님이 확인 문항을 정답/오답으로 정함 → 점수표와 빨간펜 시험지를 다시 계산해서 돌려줌"""
    if run_id not in RUNS:
        raise HTTPException(404, '채점 결과를 찾을 수 없습니다 (노트북이 다시 시작됐을 수 있어요)')
    R = RUNS[run_id]
    row = R['res'][(R['res']['시험지'] == d.sheet_id) & (R['res']['문항'] == d.question)]
    if row.empty or R['res'].loc[row.index[0], '판정'] != '교사 확인':
        raise HTTPException(422, '선생님 확인 문항이 아닙니다')
    if d.decision not in ('correct', 'wrong', 'undo'):
        raise HTTPException(422, "decision은 correct, wrong, undo 중 하나")
    with API_LOCK, use_subject(R['subject']):
        R['res'].loc[row.index[0], '선생님'] = '' if d.decision == 'undo' else KO[d.decision]
        return render_run(run_id, [d.sheet_id])

@app.get('/v1/jobs/{job_id}')
def job(job_id: str):
    if job_id not in JOBS:
        raise HTTPException(404, '채점 작업을 찾을 수 없습니다 (노트북이 다시 시작됐을 수 있어요)')
    return JOBS[job_id]

print('노트북 API 준비 완료. 다음 셀에서 서버를 켭니다.')

### 5-3. 서버 켜고 시연 주소 받기
아래에 나오는 **https://….trycloudflare.com**이 시연 주소입니다. 휴대폰이나 다른 컴퓨터에서 열어 보세요. 노트북을 다시 시작하면 주소가 바뀝니다.

In [ ]:
# 서버 켜기 + 바깥에서 들어올 수 있는 https 주소 (Cloudflare 임시 터널, 계정 필요 없음)
import uvicorn, subprocess, queue, urllib.request
if 'api_server' not in globals():
    api_server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning'))
    threading.Thread(target=api_server.run, daemon=True).start()
    for _ in range(30):
        try:
            urllib.request.urlopen('http://localhost:8000/v1/config', timeout=2); break
        except Exception:
            time.sleep(1)
print('API 서버 켜짐 (노트북 안에서 실행 중)')
if not os.path.exists('cloudflared'):
    os.system('wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/'
              'cloudflared-linux-amd64 && chmod +x cloudflared')
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stderr=subprocess.PIPE, text=True)
urls = queue.Queue()
def _drain(p):   # 터널 기록을 계속 읽어 줘야 터널이 멈추지 않음
    for line in p.stderr:
        if m := re.search(r'https://[\w-]+\.trycloudflare\.com', line):
            urls.put(m.group())
threading.Thread(target=_drain, args=(tunnel,), daemon=True).start()
try:
    API_URL = urls.get(timeout=90)
    print('시연 주소:', API_URL)
    print('웹페이지를 따로 올렸다면 페이지 주소 뒤에:  ?api=' + API_URL)
except queue.Empty:   # 임시 터널이 막히면 Colab 자체 주소로 (이 Colab에 로그인한 사람만 열 수 있음)
    try:
        from google.colab.output import eval_js
        API_URL = eval_js('google.colab.kernel.proxyPort(8000)')
        print('⚠️ 공개 주소를 만들지 못해 Colab 내부 주소를 씁니다 (내 계정에서만 열림, 테스트용):', API_URL)
    except Exception:
        print('⚠️ 공개 주소를 만들지 못했습니다. 이 셀을 다시 실행해 보세요.')
print('이 노트북(런타임)이 켜져 있는 동안만 동작합니다. 4-1에서 올린 답안지가 웹페이지의 예시 답안지로 나옵니다.')

### 5-4. 베이스라인 표 (Gemini vs Claude)
예시 답안지를 **Gemini와 Claude가 읽은 결과**를 학생이 **실제로 쓴 답**(`정답_실제.csv`, 과목_자료.zip 안에 있음)과 비교합니다: 글자 그대로 맞힌 비율, 글자 오류율(CER), 그 읽기로 판정했을 때 자동 처리 비율과 오류.
- Claude도 비교하려면 보안 비밀에 `CLAUDE_API_KEY`를 추가하세요(유료). 없으면 Gemini만 나옵니다.
- **직접 쓴 손글씨 답안지**는 4-1에 올리고, `정답_실제.csv`에 같은 형식(과목, 시험지, 반, 번호, 이름, 문항, 실제답, 정답여부, 메모)으로 행을 더해 다시 실행하세요. 보고서에는 합성 답안지와 직접 쓴 답안지 결과를 나눠 적으세요.

In [ ]:
# 베이스라인 표: 같은 답안지를 Gemini와 Claude가 읽은 결과를 학생이 실제로 쓴 답(정답_실제.csv)과 비교
#   글자 그대로 맞힘 = AI가 읽은 글자가 실제로 쓴 글자와 똑같은 비율 (띄어쓰기·대소문자 무시)
#   CER = 틀린 글자 수 / 실제 글자 수 (낮을수록 좋음),  자동 처리·오류 = 그 읽기로 판정했을 때
import base64
CLAUDE_MODEL = 'claude-haiku-5-5'   # Claude API 모델 이름 (https://docs.claude.com 에서 확인). 키가 없으면 Gemini만
TRUTH_PATH = 'inputs/subjects/정답_실제.csv'

TRUTH_DF = pd.read_csv(TRUTH_PATH, dtype=str).fillna('') if os.path.exists(TRUTH_PATH) else None
if TRUTH_DF is None:
    print('정답_실제.csv가 없습니다 (과목_자료.zip 안에 있음). 직접 쓴 답안지는 같은 형식으로 행을 추가하세요.')
else:
    claude = None
    try:
        from google.colab import userdata
        import anthropic
        claude = anthropic.Anthropic(api_key=userdata.get('CLAUDE_API_KEY'), timeout=30)
    except Exception as e:
        print(f'Claude 키가 없어 Gemini만 비교합니다 ({type(e).__name__}). 비교하려면 보안 비밀에 CLAUDE_API_KEY 추가')
    CLAUDE_CACHE_PATH = 'work/claude_cache.json'
    CLAUDE_CACHE = json.load(open(CLAUDE_CACHE_PATH, encoding='utf-8')) if os.path.exists(CLAUDE_CACHE_PATH) else {}

    def claude_read(path):
        if path not in CLAUDE_CACHE:
            msg = claude.messages.create(model=CLAUDE_MODEL, max_tokens=800, messages=[{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png',
                                             'data': base64.b64encode(open(path, 'rb').read()).decode()}},
                {'type': 'text', 'text': SHEET_PROMPT}]}])
            text = ''.join(b.text for b in msg.content if b.type == 'text')
            CLAUDE_CACHE[path] = json.loads(re.sub(r'^```(json)?|```$', '', text.strip()).strip())
            json.dump(CLAUDE_CACHE, open(CLAUDE_CACHE_PATH, 'w', encoding='utf-8'), ensure_ascii=False)
        return CLAUDE_CACHE[path]

    rows = []
    with API_LOCK:
        for name, sub in SUBJECTS.items():
            with use_subject(name):
                for sid, path in sub['pages']:
                    subj, stem = SAMPLE_ORIGIN.get(sid, (name, sid))
                    t = TRUTH_DF[(TRUTH_DF['과목'] == subj) & (TRUTH_DF['시험지'] == stem)]
                    if t.empty:
                        continue
                    cells = {q: (f'crops/{sid}_q{q}_wide.png' if OUTSIDE.get((sid, q)) else f'crops/{sid}_q{q}.png') for q in QS}
                    if not os.path.exists(f'crops/{sid}_q1.png'):
                        crop_sheet(sid, path)
                    readers = {'Gemini': None}
                    if claude:
                        readers['Claude'] = None
                    try:
                        _, per_q, messy = read_sheet(sid, cells)   # 4-4에서 읽어 둔 결과 (없으면 지금 읽음)
                        readers['Gemini'] = (per_q, messy)
                    except (TimeoutError, NoReading) as e:
                        print(f'  {sid}: Gemini 읽기 실패 ({e})')
                    if claude:
                        try:
                            r = claude_read(compose_sheet(sid, cells))
                            get = lambda q: (r.get('answers') or {}).get(str(q), {}) or {}
                            readers['Claude'] = ({q: [str(get(q).get('text', ''))] for q in QS}, {q: bool(get(q).get('messy')) for q in QS})
                        except Exception as e:
                            print(f'  {sid}: Claude 읽기 실패 ({str(e)[:80]})')
                    for model_name, got in readers.items():
                        if got is None:
                            continue
                        per_q, messy = got
                        for _, tr in t.iterrows():
                            q = int(tr['문항'])
                            if q not in per_q:
                                continue
                            reads = per_q[q]
                            top = top_read(reads)[0]
                            real = normalize(tr['실제답'])   # 지운 글씨·메모는 빼고 최종으로 쓴 답
                            f = make_features(reads, q, messy[q])
                            p = float(model.predict(np.array([[f[k] for k in FEATURES]]), verbose=0)[0, 0])
                            judge, _ = decide(p, f, OUTSIDE.get((sid, q), False), near_miss(reads, q))
                            rows.append({'모델': model_name, '과목': subj, '시험지': stem, '문항': q, '실제답': tr['실제답'], '읽은 답': top,
                                         '글자 맞힘': top == real, 'CER': edit_distance(top, real) / max(len(real), 1),
                                         '읽은 횟수': len(reads), '판정': judge, '실제 정답': tr['정답여부'] == '1'})
    base = pd.DataFrame(rows)
    if not base.empty:   # 같은 답안지가 두 번 들어간 경우(4-1과 과목 자료에 모두 있음) 한 번만 셈
        base = base.drop_duplicates(['모델', '과목', '시험지', '문항'])
    if base.empty:
        print('비교할 답안지가 없습니다 (정답_실제.csv의 과목·시험지 이름이 예시 답안지와 같은지 확인)')
    else:
        base['자동'] = base['판정'] != '교사 확인'
        base['자동 오류'] = base['자동'] & ((base['판정'] == '정답') != base['실제 정답'])
        table = base.groupby('모델').agg(답안지=('시험지', 'nunique'), 칸=('문항', 'size'),
                                        글자_맞힘=('글자 맞힘', 'mean'), 평균_CER=('CER', 'mean'), 칸당_읽은_횟수=('읽은 횟수', 'mean'),
                                        자동_처리=('자동', 'mean'), 자동_처리_중_오류=('자동 오류', 'sum')).reset_index()
        table['글자_맞힘'] = (table['글자_맞힘'] * 100).round(1).astype(str) + '%'
        table['자동_처리'] = (table['자동_처리'] * 100).round(1).astype(str) + '%'
        table['평균_CER'] = table['평균_CER'].round(3)
        display(table)
        print('과목별 글자 맞힘:')
        display(base.pivot_table(index='과목', columns='모델', values='글자 맞힘', aggfunc='mean').mul(100).round(1))
        print('AI가 다르게 읽은 칸:')
        display(base[~base['글자 맞힘']][['모델', '과목', '시험지', '문항', '실제답', '읽은 답', '판정']])
        table.to_csv('results/베이스라인.csv', index=False, encoding='utf-8-sig')
        base.to_csv('results/베이스라인_칸별.csv', index=False, encoding='utf-8-sig')
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
        sns.barplot(data=base, x='과목', y='글자 맞힘', hue='모델', ax=axes[0], errorbar=None)
        axes[0].set(title='과목별 글자 그대로 맞힌 비율', ylabel='비율', ylim=(0, 1))
        sns.barplot(data=base, x='모델', y='CER', ax=axes[1], errorbar=None, color='#4c72b0')
        axes[1].set(title='평균 글자 오류율 CER (낮을수록 좋음)')
        plt.tight_layout(); plt.savefig('results/10_baseline.png', dpi=150); plt.show()
        print('※ 합성(손글씨 글꼴) 답안지가 섞여 있으면 실제 손글씨보다 쉬울 수 있으니, 보고서에는 직접 쓴 답안지 결과를 따로 표시하세요')